# **LLM Baseline & Comparative Evaluation**

## **Objective**

Establish a controlled baseline to evaluate the difference between a
general-purpose instruction-tuned LLM and a telecom-domain LLM

The primary comparison will be between:

- **General LLM:** `EssentialAI/rnj-1-instruct`
- **Telecom LLM:** `farbodtavakkoli/OTel-LLM-8.3B-IT`

OTel-LLM-8.3B-IT is derived from the `rnj-1-instruct` model lineage and
has undergone telecom-specific full-parameter post-training. This
provides a useful controlled comparison of the impact of
telecom-domain specialisation.

## **Progression**

### 1. Environment Setup
- Install and verify required libraries.
- Verify Python, PyTorch, CUDA, and GPU availability.
- Confirm the GPU inference environment and available resources.

### 2. Model Identification & Configuration
- Identify the two baseline LLMs: General LLM and OTel 1.0.
- Identify the independent Judge LLM used for evaluation.
- Examine model architecture, parameter count, and key configuration settings.
- Document OTel 1.0 and its relationship to its base model.
- Record relevant model and tokenizer configuration for reproducibility.
- Keep the Judge LLM and its evaluation configuration fixed throughout the baseline.

### 3. Model Loading

- Load the General LLM, OTel 1.0, and independent Judge LLM.
- Apply appropriate quantisation based on model size and available GPU memory.
- Verify successful loading and device placement for each model.
- Monitor GPU memory utilisation to confirm that the models can operate within the available hardware constraints.

### 4. Inference Pipeline
- Develop a standardised response-generation function for both baseline LLMs.
- Apply consistent generation parameters across the General LLM and OTel 1.0.
- Support chat templates where available.
- Ensure correct device placement and output decoding.
- Verify consistent response generation before running the benchmark.

### 5. Telecom Benchmark
- Develop a controlled set of telecom-domain questions.
- Use the same questions, prompts and evaluation methodology for both
  models.
- Cover areas such as 5G architecture, RAN, Core, Open RAN,
  cloud-native telecom and network operations.


### 6. Comparative Evaluation
Compare the general-purpose and telecom-domain models based on:

- Telecom factual accuracy
- Relevance
- Completeness
- Technical reasoning
- Response quality
- Hallucination / unsupported claims
- Observed domain-specific capability

## **Scope Boundary**

This module intentionally focuses on **standalone LLM capability**.

It does **not** include:

- Retrieval-Augmented Generation (RAG)
- Vector databases or embeddings
- External knowledge retrieval
- Model fine-tuning or additional training
- Model Context Protocol (MCP)
- External operational tools
- Autonomous agents or workflows

These capabilities will be introduced progressively in subsequent
modules.

## **Experimental Principle**

Both models will be evaluated under comparable conditions wherever
technically possible.

The baseline established here will provide the reference point for
measuring the incremental value of:

**LLM → LLM + RAG → LLM + RAG + MCP → Autonomous Telecom Operations**

# **Environment Setup**

- Install and verify required libraries.
- Verify Python, PyTorch, CUDA, and GPU availability.
- Confirm the GPU inference environment and available resources.

## **Installing Required Libraries**

In [ ]:
## Install libraries required for LLM inference, tokenization,
# 4-bit quantization, GPU acceleration, and Hugging Face access.

!pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    safetensors \
    huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 85.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 44.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


## **Import Required Libraries**

In [ ]:
# Import core libraries, Hugging Face utilities, and PyTorch,
# then verify the software environment and available GPU memory.

import gc
import importlib.metadata as metadata
import json
import time
import torch
import transformers
import pandas as pd
import re


from huggingface_hub import list_repo_files
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset

print("========== Core Environment ==========")
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"Accelerate    : {metadata.version('accelerate')}")
print(f"bitsandbytes  : {metadata.version('bitsandbytes')}")

print("\n========== GPU Memory Diagnostic ==========")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    free_mem, total_mem = torch.cuda.mem_get_info()

    print(f"GPU Model     : {gpu.name}")
    print(f"Total VRAM    : {total_mem / (1024 ** 3):.2f} GB")
    print(f"Free VRAM     : {free_mem / (1024 ** 3):.2f} GB")
    print(f"CUDA Version  : {torch.version.cuda}")
else:
    raise RuntimeError("CUDA accelerator is required for Module 1 execution.")

========== Core Environment ==========
PyTorch       : 2.10.0+cu128
Transformers  : 5.15.1
Accelerate    : 1.14.0
bitsandbytes  : 0.50.1

========== GPU Memory Diagnostic ==========
GPU Model     : Tesla T4
Total VRAM    : 14.56 GB
Free VRAM     : 5.50 GB
CUDA Version  : 12.8


### **Observation**

The inference environment is correctly configured with CUDA-enabled PyTorch and a Tesla T4 GPU with 14.56 GB VRAM available for model loading and inference.

# **Model Identification and Configuration**

- Identify the two baseline LLMs: General LLM and OTel 1.0.
- Identify the independent Judge LLM used for evaluation.
- Examine model architecture, parameter count, and key configuration settings.
- Document OTel 1.0 and its relationship to its base model.
- Record relevant model and tokenizer configuration for reproducibility.
- Keep the Judge LLM and its evaluation configuration fixed throughout the baseline.

## **Identify and Inspect the Telecom Domain LLM**

In [ ]:
# ---------------------------------------------------------
# Identify and inspect the OTel 1.x baseline model
# ---------------------------------------------------------

# Official OTel 1.x instruction-tuned 8.3B model.
OTEL_MODEL_ID = "farbodtavakkoli/OTel-LLM-8.3B-IT"

# Load configuration only; model weights are not downloaded.
config = AutoConfig.from_pretrained(OTEL_MODEL_ID)

print("========== OTel Model ==========")
print(f"Model ID   : {OTEL_MODEL_ID}")
print(f"Model type : {config.model_type}")

print("\n========== Model Configuration ==========")
print(f"Hidden size       : {config.hidden_size}")
print(f"Number of layers  : {config.num_hidden_layers}")
print(f"Attention heads   : {config.num_attention_heads}")
print(f"Vocabulary size   : {config.vocab_size}")

# Check the configured model data type.
if hasattr(config, "dtype"):
    print(f"Configured dtype  : {config.dtype}")
elif hasattr(config, "torch_dtype"):
    print(f"Configured dtype  : {config.torch_dtype}")

print("\nConfiguration loaded successfully.")


# ---------------------------------------------------------
# Inspect the OTel 1.x model repository
# ---------------------------------------------------------
# Inspect repository files without downloading model weights.
# This identifies weight format, sharding, and supporting files.

files = list_repo_files(OTEL_MODEL_ID)

print("\n========== OTel 1.x Repository Files ==========\n")

for file in files:
    print(file)

print(f"\nTotal repository files: {len(files)}")

config.json:   0%|          | 0.00/2.64k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


========== OTel Model ==========
Model ID   : farbodtavakkoli/OTel-LLM-8.3B-IT
Model type : gemma3_text

========== Model Configuration ==========
Hidden size       : 4096
Number of layers  : 32
Attention heads   : 32
Vocabulary size   : 128256
Configured dtype  : torch.bfloat16

Configuration loaded successfully.

========== OTel 1.x Repository Files ==========

.gitattributes
README.md
chat_template.jinja
config.json
generation_config.json
pytorch_model-00001-of-00007.bin
pytorch_model-00002-of-00007.bin
pytorch_model-00003-of-00007.bin
pytorch_model-00004-of-00007.bin
pytorch_model-00005-of-00007.bin
pytorch_model-00006-of-00007.bin
pytorch_model-00007-of-00007.bin
pytorch_model.bin.index.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json

Total repository files: 16


### **Observation**

OTel 1.0 is identified as a **Gemma 3 text model** with 8.3B-scale architecture, 32 layers, 4,096 hidden size, and BF16-configured weights. The checkpoint is distributed across **7 PyTorch shards**.

The `rope_parameters` warning is noted but does not prevent configuration loading. Hugging Face authentication is also optional for this inspection stage.

## **Identify and Inspect the General LLM**

In [ ]:
# ---------------------------------------------------------
# Identify and inspect the General LLM baseline
# ---------------------------------------------------------

from transformers import AutoConfig
from huggingface_hub import list_repo_files

# General-purpose baseline model.
GENERAL_MODEL_ID = "EssentialAI/rnj-1-instruct"

# Load configuration only; model weights are not downloaded.
general_config = AutoConfig.from_pretrained(GENERAL_MODEL_ID)

print("========== General LLM ==========")
print(f"Model ID   : {GENERAL_MODEL_ID}")
print(f"Model type : {general_config.model_type}")

print("\n========== Model Configuration ==========")
print(f"Hidden size       : {general_config.hidden_size}")
print(f"Number of layers  : {general_config.num_hidden_layers}")
print(f"Attention heads   : {general_config.num_attention_heads}")
print(f"Vocabulary size   : {general_config.vocab_size}")

# Check the configured model data type.
if hasattr(general_config, "dtype"):
    print(f"Configured dtype  : {general_config.dtype}")
elif hasattr(general_config, "torch_dtype"):
    print(f"Configured dtype  : {general_config.torch_dtype}")

print("\nConfiguration loaded successfully.")


# ---------------------------------------------------------
# Inspect the General LLM repository
# ---------------------------------------------------------
# Inspect repository files without downloading model weights.
# This identifies weight format, sharding, and supporting files.

general_files = list_repo_files(GENERAL_MODEL_ID)

print("\n========== General LLM Repository Files ==========\n")

for file in general_files:
    print(file)

print(f"\nTotal repository files: {len(general_files)}")

config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


========== General LLM ==========
Model ID   : EssentialAI/rnj-1-instruct
Model type : gemma3_text

========== Model Configuration ==========
Hidden size       : 4096
Number of layers  : 32
Attention heads   : 32
Vocabulary size   : 128256
Configured dtype  : torch.float32

Configuration loaded successfully.

========== General LLM Repository Files ==========

.gitattributes
LICENSE
README.md
config.json
generation_config.json
model-00001-of-00007.safetensors
model-00002-of-00007.safetensors
model-00003-of-00007.safetensors
model-00004-of-00007.safetensors
model-00005-of-00007.safetensors
model-00006-of-00007.safetensors
model-00007-of-00007.safetensors
model.safetensors.index.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json

Total repository files: 16


### **Observation**

The General LLM is also identified as a **Gemma 3 text model** with the same 32-layer, 4,096 hidden-size architecture and 128,256-token vocabulary as OTel 1.0. Its checkpoint is distributed across **7 Safetensors shards**.

The `rope_parameters` warning is noted but does not prevent configuration loading.

## **Identify and Inspect the Judge LLM**

In [ ]:
# ---------------------------------------------------------
# Identify and inspect the Independent Judge LLM
# ---------------------------------------------------------

# Independent LLM used to evaluate the benchmark responses.
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Load configuration only; model weights are not downloaded.
judge_config = AutoConfig.from_pretrained(JUDGE_MODEL_ID)

print("========== Independent Judge LLM ==========")
print(f"Model ID   : {JUDGE_MODEL_ID}")
print(f"Model type : {judge_config.model_type}")

print("\n========== Model Configuration ==========")
print(f"Hidden size       : {judge_config.hidden_size}")
print(f"Number of layers  : {judge_config.num_hidden_layers}")
print(f"Attention heads   : {judge_config.num_attention_heads}")
print(f"Vocabulary size   : {judge_config.vocab_size}")

# Check the configured model data type.
if hasattr(judge_config, "dtype"):
    print(f"Configured dtype  : {judge_config.dtype}")
elif hasattr(judge_config, "torch_dtype"):
    print(f"Configured dtype  : {judge_config.torch_dtype}")

print("\nConfiguration loaded successfully.")


# ---------------------------------------------------------
# Inspect the Judge LLM repository
# ---------------------------------------------------------
# Inspect repository files without downloading model weights.
# This identifies weight format, sharding, and supporting files.

judge_files = list_repo_files(JUDGE_MODEL_ID)

print("\n========== Judge LLM Repository Files ==========\n")

for file in judge_files:
    print(file)

print(f"\nTotal repository files: {len(judge_files)}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

========== Independent Judge LLM ==========
Model ID   : Qwen/Qwen2.5-7B-Instruct
Model type : qwen2

========== Model Configuration ==========
Hidden size       : 3584
Number of layers  : 28
Attention heads   : 28
Vocabulary size   : 152064
Configured dtype  : torch.bfloat16

Configuration loaded successfully.

========== Judge LLM Repository Files ==========

.gitattributes
LICENSE
README.md
config.json
generation_config.json
merges.txt
model-00001-of-00004.safetensors
model-00002-of-00004.safetensors
model-00003-of-00004.safetensors
model-00004-of-00004.safetensors
model.safetensors.index.json
tokenizer.json
tokenizer_config.json
vocab.json

Total repository files: 14


### Observation

The Independent Judge is **Qwen2.5-7B-Instruct**, with 28 layers, 3,584 hidden size, and a 152,064-token vocabulary. Its checkpoint is distributed across **4 Safetensors shards** and is configured for BF16.

Configuration loaded successfully with no model-specific warnings.

## **Model Configuration Comparison**

In [ ]:
# ---------------------------------------------------------
# Compare baseline model configurations
# ---------------------------------------------------------

print("========== Baseline Model Comparison ==========")

print(f"\nGeneral LLM : {GENERAL_MODEL_ID}")
print(f"OTel 1.0    : {OTEL_MODEL_ID}")
print(f"Judge LLM   : {JUDGE_MODEL_ID}")

print("\n========== Configuration Summary ==========")

print(f"General LLM type      : {general_config.model_type}")
print(f"OTel 1.0 type         : {config.model_type}")
print(f"Judge LLM type        : {judge_config.model_type}")

print(f"\nGeneral LLM layers    : {general_config.num_hidden_layers}")
print(f"OTel 1.0 layers       : {config.num_hidden_layers}")
print(f"Judge LLM layers      : {judge_config.num_hidden_layers}")

print(f"\nGeneral LLM hidden    : {general_config.hidden_size}")
print(f"OTel 1.0 hidden       : {config.hidden_size}")
print(f"Judge LLM hidden      : {judge_config.hidden_size}")

========== Baseline Model Comparison ==========

General LLM : EssentialAI/rnj-1-instruct
OTel 1.0    : farbodtavakkoli/OTel-LLM-8.3B-IT
Judge LLM   : Qwen/Qwen2.5-7B-Instruct

========== Configuration Summary ==========
General LLM type      : gemma3_text
OTel 1.0 type         : gemma3_text
Judge LLM type        : qwen2

General LLM layers    : 32
OTel 1.0 layers       : 32
Judge LLM layers      : 28

General LLM hidden    : 4096
OTel 1.0 hidden       : 4096
Judge LLM hidden      : 3584


### Observation

OTel 1.0 and the General LLM share the same **Gemma 3 architecture, 32 layers, and 4,096 hidden size**, while the independent Judge uses a different **Qwen2 architecture**. This supports using the Judge as an architecturally independent evaluator.

# **Model Loading**

- Load the General LLM, OTel 1.0, and independent Judge LLM.
- Apply appropriate quantisation based on model size and available GPU memory.
- Verify successful loading and device placement for each model.
- Monitor GPU memory utilisation to confirm that the models can operate within the available hardware constraints.

## **Load the OTEL LLM**

In [ ]:
# ---------------------------------------------------------
# Load OTel 1.0 in 4-bit
# ---------------------------------------------------------

# 4-bit NF4 quantisation reduces memory usage for the 8.3B model.
# FP16 computation is used for compatibility with the Tesla T4.

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer.
otel_tokenizer = AutoTokenizer.from_pretrained(OTEL_MODEL_ID)

# Load quantised model with automatic device placement.
otel_model = AutoModelForCausalLM.from_pretrained(
    OTEL_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

otel_model.eval()

print("========== OTel 1.0 ==========")
print("Model loaded successfully.")
print(f"Model device: {otel_model.device}")

# Check GPU memory after loading.
if torch.cuda.is_available():
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"GPU VRAM used : {(total_mem - free_mem) / (1024 ** 3):.2f} GB")
    print(f"GPU VRAM free : {free_mem / (1024 ** 3):.2f} GB")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

...

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/419 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

========== OTel 1.0 ==========
Model loaded successfully.
Model device: cuda:0
GPU VRAM used : 3.13 GB
GPU VRAM free : 11.43 GB


### Observation

OTel 1.0 loaded successfully on the Tesla T4 and uses approximately **3.13 GB of VRAM** under 4-bit quantisation, leaving **11.43 GB available**.

The `rope_parameters` warning did not prevent successful model loading.

## **Load the General LLM**

In [ ]:
# ---------------------------------------------------------
# Load General LLM in 4-bit
# ---------------------------------------------------------

# 4-bit NF4 quantisation reduces memory usage for the model.
# FP16 computation is used for compatibility with the Tesla T4.

general_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer.
general_tokenizer = AutoTokenizer.from_pretrained(
    GENERAL_MODEL_ID
)

# Load quantised model with automatic device placement.
general_model = AutoModelForCausalLM.from_pretrained(
    GENERAL_MODEL_ID,
    quantization_config=general_quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

general_model.eval()

print("========== General LLM ==========")
print("Model loaded successfully.")
print(f"Model device: {general_model.device}")

# Check GPU memory after loading.
if torch.cuda.is_available():
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"GPU VRAM used : {(total_mem - free_mem) / (1024 ** 3):.2f} GB")
    print(f"GPU VRAM free : {free_mem / (1024 ** 3):.2f} GB")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

========== General LLM ==========
Model loaded successfully.
Model device: cuda:0
GPU VRAM used : 5.11 GB
GPU VRAM free : 9.46 GB


### Observation

The General LLM loaded successfully on the Tesla T4 using 4-bit quantisation, consuming in addition to previously loaded OTel LLM approximately **5.11 GB of VRAM** and leaving **9.46 GB available**. The 33.2 GB checkpoint was successfully loaded despite its larger on-disk size.

The `rope_parameters` warning did not prevent successful model loading.


## **Load the Judge LLM**

In [ ]:
# ---------------------------------------------------------
# Load Independent Judge LLM in 4-bit
# ---------------------------------------------------------

# 4-bit NF4 quantisation reduces memory usage.
# FP16 computation is used for Tesla T4 compatibility.

judge_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer.
judge_tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL_ID
)

# Load quantised model with automatic device placement.
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    quantization_config=judge_quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

judge_model.eval()

print("========== Independent Judge LLM ==========")
print("Model loaded successfully.")
print(f"Model device: {judge_model.device}")

# Check GPU memory after loading.
if torch.cuda.is_available():
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"GPU VRAM used : {(total_mem - free_mem) / (1024 ** 3):.2f} GB")
    print(f"GPU VRAM free : {free_mem / (1024 ** 3):.2f} GB")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

========== Independent Judge LLM ==========
Model loaded successfully.
Model device: cuda:0
GPU VRAM used : 7.06 GB
GPU VRAM free : 7.51 GB


### Observation

The Independent Judge LLM loaded successfully on the Tesla T4 using 4-bit quantisation, consuming in addition to previously loaded LLMs approximately **7.06 GB of VRAM** and leaving **7.51 GB available**. The 7B checkpoint was successfully loaded across 4 Safetensors shards.

The three models have now been successfully loaded and are available for inference.

# **Model Smoke Test**
- Test OTel 1.0
- Test General LLM
- Test Judge LLM

## **Smoke Test Function**

In [ ]:
# ---------------------------------------------------------
# Chat Template Smoke Test
# ---------------------------------------------------------

def smoke_test(model, tokenizer, prompt, model_name):
    """Verify chat-template formatting and basic generation."""

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    # Use the model's native chat template when available.
    if tokenizer.chat_template:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        formatted_prompt = prompt

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )

    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )

    print(f"\n========== {model_name} ==========")
    print(response.strip())

    return response.strip()

## **Smoke Test Execution**

In [ ]:
TEST_PROMPT = "What is the role of the AMF in a 5G Standalone network?"

general_test_response = smoke_test(
    general_model,
    general_tokenizer,
    TEST_PROMPT,
    "General LLM",
)

otel_test_response = smoke_test(
    otel_model,
    otel_tokenizer,
    TEST_PROMPT,
    "OTel 1.0",
)

judge_test_response = smoke_test(
    judge_model,
    judge_tokenizer,
    TEST_PROMPT,
    "Independent Judge LLM",
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



========== General LLM ==========
In a 5G Standalone (5G-SA) network, the Access and Mobility Function (AMF) plays a critical role in managing the mobility and access aspects of the network. The AMF is one of the key components of the 5G core network architecture, and it is responsible for several essential functions:

1. **Mobility Management**: The AMF is responsible for managing the mobility of user equipment (UE) within the network. This includes functions such as tracking area codes (TACs), location area codes (LACs), and paging of UEs. It also handles handovers between different cells and ensures that the UE maintains its context and session continuity during mobility.

2. **Authentication and Authorization**: The AMF

========== OTel 1.0 ==========
The Access and Mobility Function (AMF) is a critical component of the 5G Core Network (5GC) in a 5G Standalone architecture. It is responsible for managing access and mobility services, including user authentication, registration, an

### **Observation**

All three models successfully generated coherent responses using their native chat templates. The models are ready for the standardised inference pipeline.

# **Inference Pipeline**

- Develop a standardised response-generation function for the General LLM and OTel 1.0.
- Apply each model's native chat template where available.
- Use consistent generation parameters across both baseline models.
- Ensure correct device placement and output decoding.
- Verify reproducible response generation before benchmark execution.

## **Response Function**

In [ ]:
# ---------------------------------------------------------
# Standardised Response Generation
# ---------------------------------------------------------

def generate_response(
    model,
    tokenizer,
    question,
    max_new_tokens=500,
    do_sample=False,
    temperature=0.01,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
):
    """Generate a standardised response from a loaded LLM."""

    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    # Use the model's native chat template when available.
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = question

    # Tokenise the formatted prompt.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Place inputs on the model's execution device.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Keep sampling parameters explicit but only apply them
    # when sampling is enabled.
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
    }

    if do_sample:
        generation_kwargs.update({
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
        })

    # Generate response without gradient calculation.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    # Decode only newly generated tokens.
    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )

    return response.strip()

## **Test Response Function**

In [ ]:
# ---------------------------------------------------------
# Test Standardised Inference Pipeline
# ---------------------------------------------------------

TEST_QUESTION = (
    "What is the role of the AMF in a 5G Standalone network?"
)

print("=" * 80)
print("GENERAL LLM RESPONSE")
print("=" * 80)

general_response = generate_response(
    model=general_model,
    tokenizer=general_tokenizer,
    question=TEST_QUESTION,
)

print(general_response)


print("\n" + "=" * 80)
print("OTEL 1.0 RESPONSE")
print("=" * 80)

otel_response = generate_response(
    model=otel_model,
    tokenizer=otel_tokenizer,
    question=TEST_QUESTION,
)

print(otel_response)

GENERAL LLM RESPONSE
In a 5G Standalone (5G-SA) network, the Access and Mobility Function (AMF) plays a critical role as part of the 5G Core Network. The AMF is responsible for several key functions that enable the operation of a 5G network:

### 1. **Mobility Management**
   - The AMF manages the mobility of user equipment (UE), including handovers between cells within the same or different locations.
   - It tracks the UE's location and ensures seamless service continuity during mobility events.

### 2. **Authentication and Authorization**
   - The AMF authenticates users and authorizes access to the 5G network based on security credentials provided by the user.
   - It interacts with the Authentication, Authorization, and Security Context (AAA) server to verify user identities and permissions.

### 3. **Session Management**
   - The AMF controls the creation, modification, and termination of sessions for UEs.
   - It allocates resources such as IP addresses and manages Quality of Se

### Observation

The response function executed successfully for both the General LLM and OTel 1.0, correctly applying the chat templates, generation parameters, tokenisation, inference, and response decoding. Both models returned valid responses, confirming the function is ready for benchmark execution.

# **Telecom Benchmark Evaluation**

## **Track 1 — Custom Telecom Benchmark**
- Evaluate the General LLM and OTel 1.0 against the 20-question custom benchmark.
- Cover conceptual, procedural, troubleshooting, design and applied engineering scenarios.
- Generate responses using the fixed inference pipeline.
- Evaluate both responses using the independent Judge LLM.
- Produce a question-level scorecard and overall comparison.

### **Custom Telecom Benchmark Question Bank**

In [ ]:
# ---------------------------------------------------------
# Track 1: Custom Telecom Benchmark Question Bank
# ---------------------------------------------------------
# Questions are maintained separately from the execution code
# so they can be updated without changing the benchmark pipeline.
#
# expected_points are used only by the Judge/evaluation stage.
# They are NOT included in prompts sent to the baseline models.
# ---------------------------------------------------------

benchmark_questions = [

    # -----------------------------------------------------
    # 5G CORE
    # -----------------------------------------------------

    {
        "id": "Q01",
        "category": "5G Core",
        "question": (
            "What are the primary responsibilities of the AMF "
            "in a 5G Standalone network?"
        ),
        "expected_points": [
            "Access and mobility management",
            "UE registration management",
            "NAS signalling termination",
            "UE authentication and security context handling",
            "Mobility and reachability management",
            "Interaction with other 5G Core network functions",
        ],
    },

    {
        "id": "Q02",
        "category": "5G Core",
        "question": (
            "Explain the differences between the AMF, SMF and UPF "
            "in the 5G Core."
        ),
        "expected_points": [
            "AMF performs access and mobility management",
            "AMF handles UE registration and NAS signalling",
            "SMF manages PDU sessions",
            "SMF performs UPF selection and control",
            "UPF performs user-plane packet forwarding",
            "UPF provides connectivity toward external data networks",
            "Clear control-plane versus user-plane distinction",
        ],
    },

    # -----------------------------------------------------
    # 5G RAN
    # -----------------------------------------------------

    {
        "id": "Q03",
        "category": "5G RAN",
        "question": (
            "Explain the roles of the CU, DU and RU in a "
            "5G RAN architecture."
        ),
        "expected_points": [
            "CU performs higher-layer RAN functions",
            "DU performs lower-layer and real-time processing",
            "RU handles radio-frequency and lower-PHY functions",
            "Functional split between CU, DU and RU",
            "Relevant interfaces such as F1 and fronthaul",
            "Relationship between the three components",
        ],
    },

    {
        "id": "Q04",
        "category": "5G RAN",
        "question": (
            "What are the key differences between 4G LTE and "
            "5G NR RAN architecture?"
        ),
        "expected_points": [
            "5G NR introduces new radio and numerology capabilities",
            "5G supports wider channel bandwidths",
            "Massive MIMO and advanced beamforming",
            "Flexible functional split options",
            "CU/DU architectural decomposition",
            "5G supports NSA and SA deployment models",
            "Differences in latency, capacity and deployment flexibility",
        ],
    },

    # -----------------------------------------------------
    # 5G STANDALONE PROCEDURES
    # -----------------------------------------------------

    {
        "id": "Q05",
        "category": "5G SA Procedures",
        "question": (
            "Describe the major steps involved when a UE registers "
            "with a 5G Standalone network."
        ),
        "expected_points": [
            "UE establishes radio connection with the gNB",
            "Initial NAS registration procedure",
            "AMF selection and interaction",
            "UE authentication",
            "Subscription and security context handling",
            "Registration accept and completion",
            "Establishment of a usable UE context",
        ],
    },

    {
        "id": "Q06",
        "category": "5G SA Procedures",
        "question": (
            "Explain the process of establishing a PDU session "
            "in a 5G Standalone network."
        ),
        "expected_points": [
            "UE requests PDU session establishment",
            "AMF handles signalling and interacts with SMF",
            "SMF performs session management",
            "SMF selects and controls the UPF",
            "Policy and subscription information may influence the session",
            "IP address or other addressing configuration",
            "User-plane path is established through the UPF",
            "N3 connectivity between gNB and UPF",
        ],
    },

    # -----------------------------------------------------
    # OPEN RAN
    # -----------------------------------------------------

    {
        "id": "Q07",
        "category": "Open RAN",
        "question": (
            "What are the O-RAN O-CU, O-DU and O-RU, and how "
            "do they interact?"
        ),
        "expected_points": [
            "O-CU provides higher-layer centralized RAN functions",
            "O-DU provides distributed lower-layer processing",
            "O-RU performs radio and lower-PHY functions",
            "Functional decomposition of the RAN",
            "Open interfaces between components",
            "F1 interface between CU and DU",
            "Open Fronthaul between DU and RU",
        ],
    },

    {
        "id": "Q08",
        "category": "Open RAN",
        "question": (
            "What is the role of the RIC in an Open RAN architecture? "
            "Distinguish between the Near-RT RIC and Non-RT RIC."
        ),
        "expected_points": [
            "RIC provides intelligent control and optimisation",
            "Near-RT RIC operates on near-real-time timescales",
            "Near-RT RIC supports xApps",
            "Non-RT RIC operates on longer timescales",
            "Non-RT RIC supports rApps",
            "Policy, optimisation and analytics functions",
            "Relationship with SMO and RAN elements",
        ],
    },

    # -----------------------------------------------------
    # CLOUD-NATIVE TELECOM
    # -----------------------------------------------------

    {
        "id": "Q09",
        "category": "Cloud-Native Telecom",
        "question": (
            "What are the benefits and challenges of deploying "
            "5G Core Network Functions as cloud-native workloads?"
        ),
        "expected_points": [
            "Scalability and elasticity",
            "Automation and orchestration",
            "Containerisation",
            "Improved resource utilisation",
            "CI/CD and lifecycle automation",
            "Resilience and high availability",
            "Latency and performance challenges",
            "Distributed-system complexity",
            "Observability and troubleshooting",
            "Security and operational complexity",
        ],
    },

    {
        "id": "Q10",
        "category": "Cloud-Native Telecom",
        "question": (
            "How can Kubernetes support the deployment and "
            "lifecycle management of telecom Network Functions?"
        ),
        "expected_points": [
            "Container orchestration",
            "Automated deployment and scheduling",
            "Scaling and self-healing",
            "Service discovery and networking",
            "Rolling updates and lifecycle management",
            "Resource management",
            "High availability",
            "Integration with CNFs and cloud-native infrastructure",
            "Observability and operational tooling",
        ],
    },

    # -----------------------------------------------------
    # APPLIED TELECOM ENGINEERING
    # -----------------------------------------------------

    {
        "id": "Q11",
        "category": "Applied Telecom Engineering",
        "question": (
            "Design a 5G Radio site capable of delivering a peak UE "
            "downlink throughput of 1.5 Gbps and uplink throughput "
            "of 250 Mbps. Propose suitable spectrum bands and channel "
            "bandwidths, radio configuration, antenna configuration, "
            "vendor radio and baseband options, and the key system "
            "and network requirements needed to achieve these targets. "
            "State your assumptions and explain the engineering "
            "trade-offs behind your design."
        ),
        "expected_points": [
            "Appropriate 5G NR spectrum selection",
            "Appropriate channel bandwidth",
            "TDD/FDD consideration",
            "MIMO configuration",
            "Antenna configuration",
            "Modulation considerations",
            "Carrier aggregation where appropriate",
            "UE capability requirements",
            "Radio/baseband capacity requirements",
            "Transport/backhaul capacity",
            "5G Core capability",
            "Vendor equipment considerations",
            "Coverage versus capacity trade-offs",
            "Peak versus sustained throughput distinction",
            "Explicit engineering assumptions",
        ],
    },

    {
        "id": "Q12",
        "category": "Applied Telecom Engineering",
        "question": (
            "A 5G Standalone network has excellent RSRP and SINR, "
            "but users are achieving only 400 Mbps downlink throughput "
            "when the site is expected to deliver peak UE throughput "
            "above 1 Gbps. Develop a systematic troubleshooting approach. "
            "Identify the UE, RAN, transport, 5G Core and configuration "
            "parameters you would investigate, and explain how you "
            "would isolate the bottleneck."
        ),
        "expected_points": [
            "Verify UE capability and supported NR bands",
            "Verify MIMO capability and number of layers",
            "Check modulation and coding",
            "Check bandwidth and carrier configuration",
            "Check carrier aggregation where applicable",
            "Check PRB utilisation and cell loading",
            "Check scheduler configuration",
            "Check radio/baseband performance",
            "Check transport/backhaul capacity",
            "Check latency, packet loss and congestion",
            "Check UPF and 5G Core performance",
            "Check QoS and policy configuration",
            "Compare expected versus actual throughput at each layer",
            "Use controlled testing to isolate the bottleneck",
        ],
    },

    # -----------------------------------------------------
    # Q13-Q20 — APPLICATION / TROUBLESHOOTING
    # -----------------------------------------------------

    {
        "id": "Q13",
        "category": "5G Capacity Planning",
        "question": (
            "A 5G site has 100 MHz of TDD spectrum at 3.5 GHz and "
            "64T64R massive MIMO. Estimate the peak downlink throughput "
            "that could theoretically be achieved for a single UE. "
            "State your assumptions regarding modulation, coding, "
            "MIMO layers and spectral efficiency, and explain why "
            "field throughput would be lower."
        ),
        "expected_points": [
            "100 MHz bandwidth",
            "3.5 GHz TDD operation",
            "MIMO layer assumptions",
            "Modulation and coding assumptions",
            "Spectral-efficiency assumptions",
            "Theoretical versus practical throughput",
            "TDD overhead",
            "Control/reference-signal overhead",
            "Radio conditions and implementation losses",
        ],
    },

    {
        "id": "Q14",
        "category": "5G UL/DL Trade-off",
        "question": (
            "A 5G SA operator needs to support both 1 Gbps peak DL "
            "and 200 Mbps peak UL on a 3.5 GHz TDD network. Propose "
            "an appropriate TDD configuration and explain the "
            "trade-offs between DL capacity, UL capacity, latency "
            "and coverage."
        ),
        "expected_points": [
            "Appropriate TDD DL/UL allocation",
            "DL versus UL capacity trade-off",
            "UL coverage limitations",
            "Latency implications",
            "TDD switching considerations",
            "Traffic asymmetry assumptions",
            "Interference considerations",
        ],
    },

    {
        "id": "Q15",
        "category": "5G Throughput Troubleshooting",
        "question": (
            "A 5G SA UE has excellent RSRP and SINR but achieves only "
            "300 Mbps DL when the expected peak throughput is above "
            "1 Gbps. Develop a systematic troubleshooting methodology "
            "covering UE capability, NR configuration, scheduler, "
            "MIMO, PRBs, transport, UPF and 5GC. Explain how you would "
            "isolate the bottleneck."
        ),
        "expected_points": [
            "UE capability verification",
            "NR bandwidth and carrier configuration",
            "MIMO layers",
            "Modulation and coding",
            "PRB allocation and utilisation",
            "Scheduler behaviour",
            "Transport/backhaul capacity",
            "UPF performance",
            "5GC performance",
            "QoS and policy",
            "Controlled isolation methodology",
        ],
    },

    {
        "id": "Q16",
        "category": "Open RAN Deployment",
        "question": (
            "Design a high-level Open RAN deployment for an urban "
            "macro site using O-RU, O-DU and O-CU. Specify the key "
            "interfaces, fronthaul requirements, synchronization "
            "requirements, transport considerations and the major "
            "deployment trade-offs compared with an integrated RAN "
            "solution."
        ),
        "expected_points": [
            "O-RU, O-DU and O-CU placement",
            "F1 interface",
            "Open Fronthaul",
            "Fronthaul bandwidth and latency",
            "Timing and synchronisation",
            "Transport requirements",
            "Compute requirements",
            "Interoperability considerations",
            "Operational complexity",
            "Integrated versus disaggregated RAN trade-offs",
        ],
    },

    {
        "id": "Q17",
        "category": "Cloud-Native Telecom",
        "question": (
            "An operator wants to deploy 5G Core and selected RAN "
            "network functions on Kubernetes. Propose an architecture "
            "covering Kubernetes nodes, networking, storage, "
            "observability, high availability and workload placement. "
            "Identify which telecom workloads are suitable for "
            "cloud-native deployment and which require special "
            "consideration for real-time performance."
        ),
        "expected_points": [
            "Kubernetes cluster architecture",
            "Worker-node design",
            "Container networking",
            "Storage considerations",
            "Observability",
            "High availability",
            "Workload placement",
            "CPU and NUMA considerations",
            "Real-time workload considerations",
            "CNF deployment and lifecycle",
        ],
    },

    {
        "id": "Q18",
        "category": "RAN Capacity Expansion",
        "question": (
            "A 5G site is experiencing increasing congestion during "
            "the busy hour. The operator has 100 MHz at 3.5 GHz but "
            "cannot obtain additional spectrum. Propose at least "
            "three engineering approaches to increase capacity. "
            "Compare their expected benefits, costs, complexity and "
            "impact on coverage."
        ),
        "expected_points": [
            "Additional carrier or spectrum options where available",
            "Massive MIMO optimisation",
            "Sectorisation",
            "Small-cell densification",
            "Beamforming optimisation",
            "Load balancing",
            "Scheduler optimisation",
            "Carrier aggregation considerations",
            "Capacity versus coverage trade-offs",
            "Cost and implementation complexity",
        ],
    },

    {
        "id": "Q19",
        "category": "Network Failure Isolation",
        "question": (
            "Following a software upgrade, 5G users experience "
            "intermittent PDU session drops across multiple sites. "
            "RAN radio indicators remain normal. Develop a "
            "fault-isolation methodology across RAN, transport, "
            "5GC, Kubernetes/platform infrastructure and software "
            "configuration. Explain the evidence you would collect "
            "before identifying the root cause."
        ),
        "expected_points": [
            "Correlate issue with software upgrade",
            "RAN signalling verification",
            "Transport packet loss and latency checks",
            "AMF/SMF/UPF investigation",
            "PDU session signalling analysis",
            "Kubernetes health and resource checks",
            "Pod/container restart and event analysis",
            "Configuration and version comparison",
            "Logs, metrics and traces",
            "Rollback or controlled comparison",
            "Evidence-based root-cause isolation",
        ],
    },

    {
        "id": "Q20",
        "category": "End-to-End Network Design",
        "question": (
            "Design an end-to-end 5G SA network for an enterprise "
            "requiring high throughput, low latency and reliable "
            "connectivity. Cover the RAN, transport, 5G Core, UPF "
            "placement, cloud infrastructure, security, synchronization, "
            "observability and operational automation. State your "
            "assumptions and identify the principal engineering "
            "trade-offs."
        ),
        "expected_points": [
            "5G SA RAN architecture",
            "Transport design",
            "5G Core architecture",
            "UPF placement and latency",
            "Cloud infrastructure",
            "Security architecture",
            "Synchronization",
            "Observability",
            "Automation and lifecycle management",
            "Reliability and high availability",
            "Explicit assumptions",
            "Engineering trade-offs",
        ],
    },
]

### **Validate Custom Telecom Benchmark**

In [ ]:
# ---------------------------------------------------------
# Validate and Freeze Custom Benchmark
# ---------------------------------------------------------

print("=" * 80)
print("CUSTOM TELECOM BENCHMARK VALIDATION")
print("=" * 80)

assert len(benchmark_questions) == 20, (
    f"Expected 20 questions, found {len(benchmark_questions)}"
)

question_ids = [q["id"] for q in benchmark_questions]

assert len(set(question_ids)) == 20, "Duplicate question IDs detected."

assert question_ids == [f"Q{i:02d}" for i in range(1, 21)], (
    "Question IDs must run sequentially from Q01 to Q20."
)

required_fields = {
    "id",
    "category",
    "question",
    "expected_points",
}

for item in benchmark_questions:
    missing = required_fields - item.keys()

    assert not missing, (
        f"{item['id']} is missing: {missing}"
    )

print(f"\nTotal questions: {len(benchmark_questions)}")

print("\n========== Category Distribution ==========")

category_counts = {}

for item in benchmark_questions:
    category = item["category"]
    category_counts[category] = (
        category_counts.get(category, 0) + 1
    )

for category, count in category_counts.items():
    print(f"{category:<35} {count}")

print("\n========== Question Register ==========")

for item in benchmark_questions:
    print(
        f"{item['id']} | "
        f"{item['category']} | "
        f"{item['question']}"
    )

print("\nBenchmark validation successful.")
print("Custom Telecom Benchmark is ready for execution.")

CUSTOM TELECOM BENCHMARK VALIDATION

Total questions: 20

========== Category Distribution ==========
5G Core                             2
5G RAN                              2
5G SA Procedures                    2
Open RAN                            2
Cloud-Native Telecom                3
Applied Telecom Engineering         2
5G Capacity Planning                1
5G UL/DL Trade-off                  1
5G Throughput Troubleshooting       1
Open RAN Deployment                 1
RAN Capacity Expansion              1
Network Failure Isolation           1
End-to-End Network Design           1

========== Question Register ==========
Q01 | 5G Core | What are the primary responsibilities of the AMF in a 5G Standalone network?
Q02 | 5G Core | Explain the differences between the AMF, SMF and UPF in the 5G Core.
Q03 | 5G RAN | Explain the roles of the CU, DU and RU in a 5G RAN architecture.
Q04 | 5G RAN | What are the key differences between 4G LTE and 5G NR RAN architecture?
Q05 | 5G SA Proced

#### **Observation**

- **20 questions validated successfully.**
- Covers **5G Core, RAN, SA, Open RAN, Cloud-Native and applied engineering**.
- Includes both **conceptual and engineering/application-focused** questions.
- Benchmark is ready for execution.

### **Custom Telecom Benchmark Inference**

#### **Inference**

In [ ]:
# ---------------------------------------------------------
# Track 1 — Custom Telecom Benchmark Inference
# ---------------------------------------------------------
# Run Q01-Q20 through the General LLM and OTel 1.0.
# Responses are displayed and stored for judging.
# ---------------------------------------------------------

benchmark_results = []

print("=" * 100)
print("CUSTOM TELECOM BENCHMARK — INFERENCE")
print("=" * 100)

print(f"\nTotal questions : {len(benchmark_questions)}")
print("Models          : General LLM + OTel 1.0")
print("Judge           : Not yet applied")

for item in benchmark_questions:

    print("\n" + "=" * 100)
    print(f"{item['id']} | {item['category']}")
    print("=" * 100)

    print("\nQUESTION")
    print("-" * 100)
    print(item["question"])

    # Generate General LLM response.
    print("\nGENERAL LLM RESPONSE")
    print("-" * 100)

    general_response = generate_response(
        model=general_model,
        tokenizer=general_tokenizer,
        question=item["question"],
    )

    print(general_response)

    # Generate OTel 1.0 response.
    print("\nOTEL 1.0 RESPONSE")
    print("-" * 100)

    otel_response = generate_response(
        model=otel_model,
        tokenizer=otel_tokenizer,
        question=item["question"],
    )

    print(otel_response)

    # Store the complete benchmark record.
    benchmark_results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_points": item.get("expected_points", []),
        "general_response": general_response,
        "otel_response": otel_response,
    })

    print("\nSTATUS: Stored successfully")


print("\n" + "=" * 100)
print("CUSTOM TELECOM BENCHMARK — INFERENCE COMPLETE")
print("=" * 100)

print(f"Questions completed : {len(benchmark_results)}")
print("Responses stored    : General LLM + OTel 1.0")
print("Next step           : Response validation and Independent Judge")

CUSTOM TELECOM BENCHMARK — INFERENCE

Total questions : 20
Models          : General LLM + OTel 1.0
Judge           : Not yet applied

Q01 | 5G Core

QUESTION
----------------------------------------------------------------------------------------------------
What are the primary responsibilities of the AMF in a 5G Standalone network?

GENERAL LLM RESPONSE
----------------------------------------------------------------------------------------------------
In a 5G Standalone (5G-SA) network, the Access and Mobility Management Function (AMF) plays a critical role in managing user mobility and access to the network. The primary responsibilities of the AMF include:

1. **Authentication and Authorization**: The AMF is responsible for authenticating users and authorizing their access to the 5G network based on security credentials provided during registration.

2. **Session Management**: It manages the creation, modification, and termination of sessions between the UE and the network, ensuri

##### **Observation — Track 1 Inference**

The **20-question Custom Telecom Benchmark (Track 1) inference** was successfully executed for the General LLM and OTel 1.0.

Responses were **displayed in the notebook and stored** for subsequent independent evaluation. The outputs show clear variation between the models, including cases of **technically questionable or incomplete responses, incorrect telecom function assignments, unsupported claims, and non-responsive OTel outputs** on some applied engineering questions.

**Status:** Track 1 inference complete. Ready for response validation and independent Judge evaluation.

#### **Inference Validation**

In [ ]:
# =============================================================================
# TRACK 1 — RESPONSE VALIDATION
# =============================================================================
# Validate that all benchmark questions were completed and that both
# General LLM and OTel 1.0 responses were generated and stored.
# =============================================================================

assert len(benchmark_results) == len(benchmark_questions)

for result in benchmark_results:

    assert result["id"]
    assert result["category"]
    assert result["question"]
    assert result["general_response"]
    assert result["otel_response"]

print("=" * 80)
print("TRACK 1 RESPONSE VALIDATION")
print("=" * 80)

print(f"Expected questions : {len(benchmark_questions)}")
print(f"Completed          : {len(benchmark_results)}")
print("General responses  : PASS")
print("OTel responses     : PASS")
print("Overall validation : PASS")

TRACK 1 RESPONSE VALIDATION
Expected questions : 20
Completed          : 20
General responses  : PASS
OTel responses     : PASS
Overall validation : PASS


##### **Observation — Track 1 Response Validation**

All **20 Track 1 benchmark questions** were completed successfully, with both General LLM and OTel 1.0 responses present and validated.

**Status:** Response validation passed. Ready for independent Judge evaluation.

### **Track 1 Independent Judge LLM Evaluation**

#### **Independent Judge LLM Prompt**

In [ ]:
# =============================================================================
# TRACK 1 — INDEPENDENT JUDGE SYSTEM PROMPT
# =============================================================================

JUDGE_SYSTEM_PROMPT = """
You are an independent senior telecommunications technical evaluator.

Evaluate two answers to the same telecommunications engineering question.
Apply the same methodology regardless of domain, including 4G, 5G, 5G Core,
RAN, Open RAN, Cloud-Native Telecom, troubleshooting, performance and design.

===============================================================================
1. CORE EVALUATION RULES
===============================================================================

- Evaluate General and OTel independently before comparing them.
- Judge technical meaning, not length, vocabulary, confidence or style.
- A verbose answer can be wrong; a concise answer can be correct.
- Evaluate only what is actually stated.
- Do not infer missing content.
- Do not reward invented, unsupported or hallucinated detail.
- Do not penalize concise answers merely for being concise.

===============================================================================
2. TECHNICAL VERIFICATION
===============================================================================

Verify material technical claims against the most authoritative technical
basis relevant to the question.

Priority:

1. Benchmark-provided reference criteria or reference material, where available.
2. Applicable 3GPP specifications.
3. Applicable ETSI specifications.
4. O-RAN Alliance specifications for Open RAN topics.
5. IETF RFCs for relevant networking/protocol topics.
6. GSMA technical publications where applicable.
7. Official Kubernetes/CNCF documentation for cloud-native topics.
8. Established telecommunications engineering principles and calculations.

Use only sources relevant to the question.

Do not treat vendor-specific behaviour as a general industry standard unless
the question explicitly concerns that vendor or implementation.

If authoritative reference material is not actually available to you, use
established technical knowledge but do not claim that an external source was
consulted.

===============================================================================
3. TECHNICAL OWNERSHIP
===============================================================================

When an answer assigns a capability or responsibility to a function,
component, protocol or architectural element, verify that the entity actually
owns that responsibility.

Distinguish between:

- owning a function;
- participating in a function;
- interacting with another function;
- providing information;
- receiving information;
- enforcing policy;
- terminating signalling;
- transporting signalling.

Do not treat:

"participates in" = "owns"
"interacts with" = "is responsible for"
"supports" = "manages"
"receives policy" = "enforces policy"
"forwards/relays" = "terminates"

A material responsibility-assignment error is a technical error.

===============================================================================
4. EXPECTED EVALUATION POINTS
===============================================================================

- Use expected_points to assess intended coverage and completeness.
- They are evaluation criteria, not necessarily a complete reference answer.
- Correctly covering expected points does not excuse additional technical errors.
- Technically valid alternative approaches should receive credit.

===============================================================================
5. RESPONSE STATUS
===============================================================================

EMPTY
- No meaningful answer.

NON_RESPONSIVE
- Does not meaningfully address the question.

PARTIAL
- Addresses the question but misses important requirements.

SUBSTANTIVE
- Meaningfully addresses the question.

===============================================================================
6. ERROR SEVERITY
===============================================================================

MINOR
- Low-impact terminology issue or imprecision.

MODERATE
- Technically incorrect claim affecting part of the answer.

MAJOR
- Fundamental misunderstanding, incorrect architecture/design, or error
  capable of leading to incorrect implementation.

===============================================================================
7. TECHNICAL ACCURACY — HOW TO SCORE
===============================================================================

First assess the material claims made by the answer.

Then assess the impact of the findings.

9-10 = EXCELLENT
- Essentially all material claims are correct.
- No material technical, architectural, procedural or quantitative errors.
- Only negligible imprecision may remain.

8 = STRONG
- Nearly all material claims are correct.
- No major technical errors.
- Only minor inaccuracies, omissions or imprecision.

7 = GOOD / STRONG WITH LIMITATIONS
- Fundamentally correct.
- Main conclusions are reliable.
- Limited omissions, imprecision or isolated errors may exist.
- No fundamental misunderstanding.

5-6 = PARTIALLY CORRECT / MIXED
- Meaningful correct information exists.
- Material omissions and/or moderate technical errors are present.
- Answer requires technical review or correction.

4 = POOR
- Significant technical deficiencies.
- Multiple important errors/omissions and/or a major weakness.

3 = VERY POOR
- Substantial misunderstanding.
- Multiple material errors.
- Most important conclusions are unreliable.

1-2 = FUNDAMENTALLY INCORRECT
- Essentially unusable, fundamentally wrong, empty or non-responsive.

===============================================================================
8. TECHNICAL ACCURACY GUARDRAILS
===============================================================================

- Technical accuracy is the dominant consideration.
- Completeness does not compensate for material technical errors.
- Verbosity does not increase the score.
- Correct terminology does not compensate for incorrect claims.
- A single MAJOR error can materially reduce the score.
- Multiple MODERATE errors should normally prevent technical_accuracy >= 8.
- Material architecture/function-ownership errors should normally prevent
  technical_accuracy >= 8.
- Scores of 9-10 should be reserved for answers without material technical
  errors.
- EMPTY/NON_RESPONSIVE answers should normally score 1-2.

===============================================================================
9. OTHER EVALUATION DIMENSIONS
===============================================================================

Score each answer from 1-10 for:

- technical_accuracy
- completeness
- relevance
- engineering_reasoning
- practical_applicability
- factual_reliability
- overall_score

overall_score is a professional judgement, not a simple arithmetic average.

Technical accuracy carries the greatest weight.

===============================================================================
10. COMPARISON
===============================================================================

Only after independently evaluating both answers:

- compare technical correctness;
- compare essential completeness;
- compare engineering reasoning where relevant;
- compare practical applicability where relevant.

Do not choose a winner merely because one answer is longer or more detailed.

Winner MUST be exactly one of:

GENERAL
OTEL
TIE

Confidence must be an integer from 0 to 100.

===============================================================================
11. OUTPUT
===============================================================================

Return ONLY valid JSON.
Do not use Markdown.
Do not use code fences.
Do not include commentary outside the JSON.
"""


#### **Independent Judge LLM Prompt**

In [ ]:
# =============================================================================
# TRACK 1 — FINAL INDEPENDENT JUDGE FUNCTION
# =============================================================================

import json
import re


def judge_responses(
    question,
    expected_points,
    general_response,
    otel_response,
):
    """Run the independent Judge and return validated JSON."""

    user_prompt = f"""
Evaluate the two answers below independently.

QUESTION:
{question}

EXPECTED EVALUATION POINTS:
{json.dumps(
    expected_points,
    ensure_ascii=False,
    indent=2,
)}

These are evaluation criteria, not necessarily a complete reference answer.

================ GENERAL LLM ANSWER ================

{general_response}

================ OTEL ANSWER ================

{otel_response}

===============================================================================
EVALUATION
===============================================================================

For each answer:

1. Determine what the question requires.
2. Determine what the answer actually claims.
3. Verify material claims against the appropriate technical basis specified
   in the system instructions.
4. Assess whether required information is present.
5. Assess the six evaluation dimensions.
6. Assign overall_score.
7. Only then compare the two answers.

For every significant technical error provide:

- claim
- severity: MINOR / MODERATE / MAJOR
- explanation

Only report material technical errors.

For missing_important_points, list only requirements that are genuinely absent.
Do not list something as missing if the answer actually addresses it, even if
the statement is technically incorrect. Technical incorrectness belongs under
technical_accuracy/significant_technical_errors.

Use the scoring framework defined in the system instructions.

IMPORTANT:

- Correct expected-point coverage does not offset material technical errors.
- A verbose answer must not receive a higher score merely for being verbose.
- Multiple moderate technical errors should normally prevent technical_accuracy
  >= 8.
- A material function-ownership or architectural error should normally prevent
  technical_accuracy >= 8.
- Do not invent errors that are not technically supported.
- EMPTY or NON_RESPONSIVE responses must be scored accordingly.

Finally compare the independently evaluated answers.

Winner MUST be exactly:
GENERAL
OTEL
TIE

Confidence MUST be an integer from 0 to 100.

Return ONLY this JSON:

{{
  "general": {{
    "response_status": "",
    "technical_accuracy": 0,
    "completeness": 0,
    "relevance": 0,
    "engineering_reasoning": 0,
    "practical_applicability": 0,
    "factual_reliability": 0,
    "overall_score": 0,
    "significant_technical_errors": [
      {{
        "claim": "",
        "severity": "",
        "explanation": ""
      }}
    ],
    "missing_important_points": []
  }},

  "otel": {{
    "response_status": "",
    "technical_accuracy": 0,
    "completeness": 0,
    "relevance": 0,
    "engineering_reasoning": 0,
    "practical_applicability": 0,
    "factual_reliability": 0,
    "overall_score": 0,
    "significant_technical_errors": [
      {{
        "claim": "",
        "severity": "",
        "explanation": ""
      }}
    ],
    "missing_important_points": []
  }},

  "comparison": {{
    "winner": "",
    "confidence": 0,
    "reason": ""
  }}
}}
"""

    # -------------------------------------------------------------------------
    # Generate Judge response
    # -------------------------------------------------------------------------

    raw_response = generate_response(
        model=judge_model,
        tokenizer=judge_tokenizer,
        question=(
            JUDGE_SYSTEM_PROMPT
            + "\n\n"
            + user_prompt
        ),
    )

    if raw_response is None:
        raise ValueError(
            "Independent Judge returned None."
        )

    cleaned = str(raw_response).strip()

    if not cleaned:
        raise ValueError(
            "Independent Judge returned an empty response."
        )

    # -------------------------------------------------------------------------
    # Remove Markdown fences if present
    # -------------------------------------------------------------------------

    cleaned = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )

    cleaned = re.sub(
        r"\s*```\s*$",
        "",
        cleaned,
    )

    cleaned = cleaned.strip()

    # -------------------------------------------------------------------------
    # Parse JSON
    # -------------------------------------------------------------------------

    try:
        judgement = json.loads(cleaned)

    except json.JSONDecodeError as e:
        raise ValueError(
            "Independent Judge returned incomplete or invalid JSON.\n\n"
            f"Parser error: {e}\n\n"
            f"Raw Judge response:\n{raw_response}"
        )

    # -------------------------------------------------------------------------
    # Validate top-level structure
    # -------------------------------------------------------------------------

    required_top_keys = {
        "general",
        "otel",
        "comparison",
    }

    missing_keys = (
        required_top_keys
        - set(judgement.keys())
    )

    if missing_keys:
        raise ValueError(
            f"Judge output missing required fields: "
            f"{sorted(missing_keys)}"
        )

    # -------------------------------------------------------------------------
    # Validate answer structure and scores
    # -------------------------------------------------------------------------

    valid_statuses = {
        "EMPTY",
        "NON_RESPONSIVE",
        "PARTIAL",
        "SUBSTANTIVE",
    }

    score_fields = [
        "technical_accuracy",
        "completeness",
        "relevance",
        "engineering_reasoning",
        "practical_applicability",
        "factual_reliability",
        "overall_score",
    ]

    for model_name in ("general", "otel"):

        result = judgement[model_name]

        # Required fields.
        required_fields = {
            "response_status",
            "technical_accuracy",
            "completeness",
            "relevance",
            "engineering_reasoning",
            "practical_applicability",
            "factual_reliability",
            "overall_score",
            "significant_technical_errors",
            "missing_important_points",
        }

        missing_fields = (
            required_fields
            - set(result.keys())
        )

        if missing_fields:
            raise ValueError(
                f"{model_name} evaluation missing fields: "
                f"{sorted(missing_fields)}"
            )

        # Response status.
        if result["response_status"] not in valid_statuses:
            raise ValueError(
                f"Invalid response_status for {model_name}: "
                f"{result['response_status']}"
            )

        # Score validation.
        for field in score_fields:

            value = result[field]

            if (
                not isinstance(value, (int, float))
                or not 1 <= value <= 10
            ):
                raise ValueError(
                    f"Invalid {field} for {model_name}: "
                    f"{value}"
                )

            result[field] = int(value)

        # Error / omission structure.
        if not isinstance(
            result["significant_technical_errors"],
            list,
        ):
            raise ValueError(
                f"{model_name}.significant_technical_errors "
                "must be a list."
            )

        if not isinstance(
            result["missing_important_points"],
            list,
        ):
            raise ValueError(
                f"{model_name}.missing_important_points "
                "must be a list."
            )

        # Validate individual errors.
        for error in result[
            "significant_technical_errors"
        ]:

            if not isinstance(error, dict):
                raise ValueError(
                    f"Invalid technical error structure for {model_name}."
                )

            required_error_fields = {
                "claim",
                "severity",
                "explanation",
            }

            missing_error_fields = (
                required_error_fields
                - set(error.keys())
            )

            if missing_error_fields:
                raise ValueError(
                    f"Technical error missing fields for {model_name}: "
                    f"{sorted(missing_error_fields)}"
                )

            if error["severity"] not in {
                "MINOR",
                "MODERATE",
                "MAJOR",
            }:
                raise ValueError(
                    f"Invalid error severity for {model_name}: "
                    f"{error['severity']}"
                )

    # -------------------------------------------------------------------------
    # Validate comparison
    # -------------------------------------------------------------------------

    comparison = judgement["comparison"]

    winner = comparison.get(
        "winner"
    )

    if winner not in {
        "GENERAL",
        "OTEL",
        "TIE",
    }:
        raise ValueError(
            f"Invalid winner returned by Judge: {winner}"
        )

    confidence = comparison.get(
        "confidence"
    )

    if (
        not isinstance(confidence, (int, float))
        or not 0 <= confidence <= 100
    ):
        raise ValueError(
            f"Invalid comparison confidence: {confidence}"
        )

    comparison["confidence"] = int(confidence)

    return judgement


#### **Independent Judge Calibration**

In [ ]:
# =============================================================================
# TRACK 1 — Q01 JUDGE CALIBRATION
# =============================================================================

q01 = benchmark_results[0]

print("=" * 80)
print("TRACK 1 — Q01 INDEPENDENT JUDGE CALIBRATION")
print("=" * 80)

print("\nQUESTION:")
print(q01["question"])

print("\nEXPECTED EVALUATION POINTS:")
print(
    json.dumps(
        q01.get("expected_points", []),
        indent=2,
        ensure_ascii=False,
    )
)

print("\nRunning independent evaluation...")

q01_judgement = judge_responses(
    question=q01["question"],
    expected_points=q01.get("expected_points", []),
    general_response=q01["general_response"],
    otel_response=q01["otel_response"],
)

print("\n========== JUDGE RESULT ==========\n")

print(
    json.dumps(
        q01_judgement,
        indent=2,
        ensure_ascii=False,
    )
)

TRACK 1 — Q01 INDEPENDENT JUDGE CALIBRATION

QUESTION:
What are the primary responsibilities of the AMF in a 5G Standalone network?

EXPECTED EVALUATION POINTS:
[
  "Access and mobility management",
  "UE registration management",
  "NAS signalling termination",
  "UE authentication and security context handling",
  "Mobility and reachability management",
  "Interaction with other 5G Core network functions"
]

Running independent evaluation...

========== JUDGE RESULT ==========

{
  "general": {
    "response_status": "SUBSTANTIVE",
    "technical_accuracy": 8,
    "completeness": 8,
    "relevance": 9,
    "engineering_reasoning": 8,
    "practical_applicability": 8,
    "factual_reliability": 9,
    "overall_score": 8,
    "significant_technical_errors": [],
    "missing_important_points": []
  },
  "otel": {
    "response_status": "SUBSTANTIVE",
    "technical_accuracy": 7,
    "completeness": 7,
    "relevance": 8,
    "engineering_reasoning": 7,
    "practical_applicability": 7,


#### **Independent Judge LLM Evaluation**

In [ ]:
# =============================================================================
# TRACK 1 — FULL INDEPENDENT JUDGE EVALUATION
# =============================================================================
# Evaluates all 20 stored Track 1 responses.
#
# The original model responses are NOT regenerated.
# Each question is evaluated by the independent Judge.
# A single controlled retry is available inside judge_responses().
# Only the final evaluation outcome is displayed.
# =============================================================================

track1_judge_results = []

print("=" * 80)
print("TRACK 1 — FULL INDEPENDENT LLM JUDGE EVALUATION")
print("=" * 80)

print(f"\nQuestions to evaluate : {len(benchmark_results)}")
print(f"Responses to evaluate : {len(benchmark_results) * 2}")
print("Evaluation method     : Independent Judge LLM")
print("Expert assessment     : Not yet applied")
print()


for i, record in enumerate(benchmark_results):

    question_id = record.get(
        "id",
        f"Q{i+1:02d}"
    )

    category = record.get(
        "category",
        "Unknown"
    )

    try:

        judgement = judge_responses(
            question=record["question"],
            expected_points=record.get(
                "expected_points",
                []
            ),
            general_response=record["general_response"],
            otel_response=record["otel_response"],
        )

        # Store the complete evaluation record.
        track1_judge_results.append({
            "id": question_id,
            "category": category,
            "question": record["question"],
            "expected_points": record.get(
                "expected_points",
                []
            ),
            "general_response": record["general_response"],
            "otel_response": record["otel_response"],
            "judge": judgement,
        })

        winner = judgement["comparison"]["winner"]
        confidence = judgement["comparison"]["confidence"]

        print(
            f"Evaluating {question_id} | {category}"
        )

        print(
            f"   ✓ Evaluation complete | "
            f"Winner: {winner} | "
            f"Confidence: {confidence}"
        )

    except Exception as e:

        # Keep failure in the stored results so it cannot be silently lost.
        track1_judge_results.append({
            "id": question_id,
            "category": category,
            "question": record["question"],
            "expected_points": record.get(
                "expected_points",
                []
            ),
            "general_response": record["general_response"],
            "otel_response": record["otel_response"],
            "judge": {
                "evaluation_error": str(e)
            },
        })

        print(
            f"Evaluating {question_id} | {category}"
        )

        print(
            "   ✗ Final evaluation failed"
        )


print()
print("=" * 80)
print("TRACK 1 — FULL INDEPENDENT JUDGE EVALUATION COMPLETE")
print("=" * 80)

successful = sum(
    1
    for result in track1_judge_results
    if "evaluation_error" not in result["judge"]
)

failed = len(track1_judge_results) - successful

print(f"Questions processed : {len(track1_judge_results)}")
print(f"Successful          : {successful}")
print(f"Failed              : {failed}")

TRACK 1 — FULL INDEPENDENT LLM JUDGE EVALUATION

Questions to evaluate : 20
Responses to evaluate : 40
Evaluation method     : Independent Judge LLM
Expert assessment     : Not yet applied

   [Judge retry] First attempt failed: Missing top-level fields: ['comparison']
Evaluating Q01 | 5G Core
   ✓ Evaluation complete | Winner: GENERAL | Confidence: 95
   [Judge retry] First attempt failed: Missing top-level fields: ['comparison']
Evaluating Q02 | 5G Core
   ✓ Evaluation complete | Winner: GENERAL | Confidence: 95
   [Judge retry] First attempt failed: Missing top-level fields: ['general', 'otel']
Evaluating Q03 | 5G RAN
   ✓ Evaluation complete | Winner: GENERAL | Confidence: 95
   [Judge retry] First attempt failed: Missing top-level fields: ['comparison', 'general', 'otel']
Evaluating Q04 | 5G RAN
   ✓ Evaluation complete | Winner: GENERAL | Confidence: 95
   [Judge retry] First attempt failed: Missing significant_technical_errors for general
Evaluating Q05 | 5G SA Procedures
   ✓ E

##### **Observation — Track 1 Independent Judge Evaluation**

The **20-question Track 1 independent Judge evaluation** was successfully completed for the General LLM and OTel 1.0.

All **40 model responses** were evaluated, with valid final Judge assessments produced for **all 20 questions**, including controlled retries where required.

**Status:** Track 1 independent Judge evaluation complete. Ready for Question-Level Scorecard and subsequent Expert Assessment.

#### **Independent Judge Response Validation**

In [ ]:
# =============================================================================
# TRACK 1 — FINAL JUDGE RESPONSE VALIDATION
# =============================================================================

print("=" * 80)
print("TRACK 1 — FINAL JUDGE RESPONSE VALIDATION")
print("=" * 80)

expected_count = len(benchmark_results)

successful = 0
failed_ids = []

for result in track1_judge_results:

    judge_result = result["judge"]

    if "evaluation_error" in judge_result:

        failed_ids.append(
            result["id"]
        )

    else:

        successful += 1


print(f"Expected questions : {expected_count}")
print(f"Judge records      : {len(track1_judge_results)}")
print(f"Successful judges  : {successful}")
print(f"Failed judges      : {len(failed_ids)}")

if not failed_ids:

    print("Overall validation : PASS")

else:

    print(
        f"Failed questions   : {failed_ids}"
    )
    print(
        "Overall validation : REVIEW REQUIRED"
    )

assert len(track1_judge_results) == expected_count
assert len(failed_ids) == 0

TRACK 1 — FINAL JUDGE RESPONSE VALIDATION
Expected questions : 20
Judge records      : 20
Successful judges  : 20
Failed judges      : 0
Overall validation : PASS


In [ ]:
# =============================================================================
# TRACK 1 — FINAL JUDGE VALIDATION
# =============================================================================

print("=" * 80)
print("TRACK 1 — FINAL INDEPENDENT JUDGE VALIDATION")
print("=" * 80)

expected_count = len(benchmark_results)

successful = 0
failed = []

for result in track1_judge_results:

    judge_result = result["judge"]

    if "evaluation_error" in judge_result:
        failed.append(result["id"])
    else:
        successful += 1


print(f"Expected questions : {expected_count}")
print(f"Judge records      : {len(track1_judge_results)}")
print(f"Successful judges  : {successful}")
print(f"Failed judges      : {len(failed)}")

if failed:
    print(f"Failed questions   : {failed}")
    print("Overall validation : REVIEW REQUIRED")
else:
    print("Overall validation : PASS")

assert len(track1_judge_results) == expected_count

TRACK 1 — FINAL INDEPENDENT JUDGE VALIDATION
Expected questions : 20
Judge records      : 20
Successful judges  : 20
Failed judges      : 0
Overall validation : PASS


##### **Observation — Track 1 Judge Response Validation**

The **final validation of the Track 1 independent Judge results** completed successfully.

All **20 expected Judge records** were present, with **20 successful evaluations and 0 failures**.

**Status:** Track 1 Judge response validation passed. Ready for the Question-Level Scorecard.

### **Track 1 Question-Level Scorecard**

In [ ]:
# =============================================================================
# TRACK 1 — QUESTION-LEVEL SCORECARD
# =============================================================================
# Converts the completed independent Judge evaluations into a question-level
# scorecard for comparison of General LLM and OTel 1.0.
# =============================================================================

import pandas as pd


scorecard_rows = []


for result in track1_judge_results:

    judge = result["judge"]

    general = judge["general"]
    otel = judge["otel"]
    comparison = judge["comparison"]

    scorecard_rows.append({
        # ---------------------------------------------------------
        # Question metadata
        # ---------------------------------------------------------
        "id": result["id"],
        "category": result["category"],

        # ---------------------------------------------------------
        # General LLM scores
        # ---------------------------------------------------------
        "general_technical_accuracy":
            general["technical_accuracy"],

        "general_completeness":
            general["completeness"],

        "general_relevance":
            general["relevance"],

        "general_engineering_reasoning":
            general["engineering_reasoning"],

        "general_practical_applicability":
            general["practical_applicability"],

        "general_factual_reliability":
            general["factual_reliability"],

        "general_overall_score":
            general["overall_score"],

        # ---------------------------------------------------------
        # OTel 1.0 scores
        # ---------------------------------------------------------
        "otel_technical_accuracy":
            otel["technical_accuracy"],

        "otel_completeness":
            otel["completeness"],

        "otel_relevance":
            otel["relevance"],

        "otel_engineering_reasoning":
            otel["engineering_reasoning"],

        "otel_practical_applicability":
            otel["practical_applicability"],

        "otel_factual_reliability":
            otel["factual_reliability"],

        "otel_overall_score":
            otel["overall_score"],

        # ---------------------------------------------------------
        # Comparison
        # ---------------------------------------------------------
        "winner":
            comparison["winner"],

        "confidence":
            comparison["confidence"],

        # ---------------------------------------------------------
        # Technical findings
        # ---------------------------------------------------------
        "general_technical_errors":
            len(general["significant_technical_errors"]),

        "otel_technical_errors":
            len(otel["significant_technical_errors"]),

        "general_missing_points":
            len(general["missing_important_points"]),

        "otel_missing_points":
            len(otel["missing_important_points"]),
    })


track1_scorecard = pd.DataFrame(
    scorecard_rows
)


# =============================================================================
# Display
# =============================================================================

print("=" * 100)
print("TRACK 1 — QUESTION-LEVEL SCORECARD")
print("=" * 100)

print(
    f"\nQuestions scored : {len(track1_scorecard)}"
)

print(
    f"Judge records    : {len(track1_judge_results)}"
)

display(
    track1_scorecard
)

TRACK 1 — QUESTION-LEVEL SCORECARD

Questions scored : 20
Judge records    : 20


,id,category,general_technical_accuracy,general_completeness,general_relevance,general_engineering_reasoning,general_practical_applicability,general_factual_reliability,general_overall_score,otel_technical_accuracy,...,otel_engineering_reasoning,otel_practical_applicability,otel_factual_reliability,otel_overall_score,winner,confidence,general_technical_errors,otel_technical_errors,general_missing_points,otel_missing_points
0,Q01,5G Core,8,8,9,8,8,9,8,7,...,7,7,8,7,GENERAL,95,0,1,0,1
1,Q02,5G Core,8,8,8,8,8,8,8,7,...,7,7,7,7,GENERAL,95,0,1,0,0
2,Q03,5G RAN,8,8,8,8,8,8,8,7,...,7,7,7,7,GENERAL,95,0,1,0,1
3,Q04,5G RAN,8,8,8,8,8,8,8,7,...,7,7,7,7,GENERAL,95,0,1,0,1
4,Q05,5G SA Procedures,4,5,8,5,6,7,6,5,...,6,7,8,7,OTEL,80,1,1,3,3
5,Q06,5G SA Procedures,8,8,8,8,8,8,8,7,...,7,7,7,7,GENERAL,95,0,1,0,1
6,Q07,Open RAN,7,7,7,7,7,7,7,8,...,8,8,8,8,OTEL,95,1,0,1,0
7,Q08,Open RAN,8,7,9,8,8,9,8,7,...,7,7,8,7,GENERAL,90,1,2,1,1
8,Q09,Cloud-Native Telecom,8,8,8,8,8,8,8,7,...,7,7,7,7,GENERAL,90,0,1,0,0
9,Q10,Cloud-Native Telecom,8,8,9,8,8,9,8,7,...,7,7,8,7,GENERAL,95,0,1,0,1


In [ ]:
# =============================================================================
# TRACK 1 — QUESTION-LEVEL SCORE SUMMARY
# =============================================================================

score_summary = track1_scorecard[
    [
        "id",
        "category",
        "general_overall_score",
        "otel_overall_score",
        "winner",
        "confidence",
    ]
].copy()


print("=" * 100)
print("TRACK 1 — QUESTION-LEVEL SCORE SUMMARY")
print("=" * 100)

display(score_summary)

TRACK 1 — QUESTION-LEVEL SCORE SUMMARY


,id,category,general_overall_score,otel_overall_score,winner,confidence
0,Q01,5G Core,8,7,GENERAL,95
1,Q02,5G Core,8,7,GENERAL,95
2,Q03,5G RAN,8,7,GENERAL,95
3,Q04,5G RAN,8,7,GENERAL,95
4,Q05,5G SA Procedures,6,7,OTEL,80
5,Q06,5G SA Procedures,8,7,GENERAL,95
6,Q07,Open RAN,7,8,OTEL,95
7,Q08,Open RAN,8,7,GENERAL,90
8,Q09,Cloud-Native Telecom,8,7,GENERAL,90
9,Q10,Cloud-Native Telecom,8,7,GENERAL,95


##### **Observation — Track 1 Overall Judge Results**

The independent Judge evaluation shows a clear performance advantage for the **General LLM** across the 20-question Track 1 benchmark.

The General LLM achieved an average overall score of **7.60/10**, compared with **6.30/10** for OTel 1.0, representing a **1.30-point advantage**. The General LLM was rated higher on **15 of 20 questions (75%)**, while OTel 1.0 was rated higher on **5 questions (25%)**.

OTel 1.0 performed comparatively better on **Q05, Q07, Q11, Q13 and Q14**, while the General LLM led across the majority of the remaining benchmark questions.

The OTel scores of **1/10 on Q16, Q19 and Q20** correspond to cases where the model did not provide a meaningful response during inference.

**Status:** Track 1 independent Judge overall comparison complete. Ready for Expert Assessment and Judge-versus-Expert analysis.

### **Track 1 Detailed Judge Evaluation Information**

In [ ]:
# =============================================================================
# TRACK 1 — DETAILED INDEPENDENT JUDGE ANALYSIS
# =============================================================================
# Displays the stored Judge evaluation for each question.
# No model inference or Judge evaluation is rerun.
# =============================================================================

for result in track1_judge_results:

    print("\n" + "=" * 100)
    print(
        f"{result['id']} | {result['category']}"
    )
    print("=" * 100)

    print("\nQUESTION")
    print("-" * 100)
    print(result["question"])

    judge = result["judge"]

    if "evaluation_error" in judge:

        print("\nJUDGE STATUS")
        print("-" * 100)
        print(
            f"ERROR: {judge['evaluation_error']}"
        )
        continue

    # -------------------------------------------------------------------------
    # General LLM assessment
    # -------------------------------------------------------------------------

    general = judge["general"]

    print("\nGENERAL LLM — JUDGE ASSESSMENT")
    print("-" * 100)

    print(
        f"Response status        : {general['response_status']}"
    )

    print(
        f"Technical accuracy     : {general['technical_accuracy']}/10"
    )

    print(
        f"Completeness           : {general['completeness']}/10"
    )

    print(
        f"Relevance              : {general['relevance']}/10"
    )

    print(
        f"Engineering reasoning  : {general['engineering_reasoning']}/10"
    )

    print(
        f"Practical applicability: "
        f"{general['practical_applicability']}/10"
    )

    print(
        f"Factual reliability    : "
        f"{general['factual_reliability']}/10"
    )

    print(
        f"Overall score          : "
        f"{general['overall_score']}/10"
    )

    print("\nSignificant technical errors:")
    print(
        json.dumps(
            general["significant_technical_errors"],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nMissing important points:")
    print(
        json.dumps(
            general["missing_important_points"],
            indent=2,
            ensure_ascii=False,
        )
    )

    # -------------------------------------------------------------------------
    # OTel assessment
    # -------------------------------------------------------------------------

    otel = judge["otel"]

    print("\nOTEL 1.0 — JUDGE ASSESSMENT")
    print("-" * 100)

    print(
        f"Response status        : {otel['response_status']}"
    )

    print(
        f"Technical accuracy     : {otel['technical_accuracy']}/10"
    )

    print(
        f"Completeness           : {otel['completeness']}/10"
    )

    print(
        f"Relevance              : {otel['relevance']}/10"
    )

    print(
        f"Engineering reasoning  : {otel['engineering_reasoning']}/10"
    )

    print(
        f"Practical applicability: "
        f"{otel['practical_applicability']}/10"
    )

    print(
        f"Factual reliability    : "
        f"{otel['factual_reliability']}/10"
    )

    print(
        f"Overall score          : "
        f"{otel['overall_score']}/10"
    )

    print("\nSignificant technical errors:")
    print(
        json.dumps(
            otel["significant_technical_errors"],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nMissing important points:")
    print(
        json.dumps(
            otel["missing_important_points"],
            indent=2,
            ensure_ascii=False,
        )
    )

    # -------------------------------------------------------------------------
    # Comparison / reasoning
    # -------------------------------------------------------------------------

    comparison = judge["comparison"]

    print("\nJUDGE COMPARISON")
    print("-" * 100)

    print(
        f"Winner     : {comparison['winner']}"
    )

    print(
        f"Confidence : {comparison['confidence']}/100"
    )

    print(
        f"Reason     : {comparison.get('reason', '')}"
    )


Q01 | 5G Core

QUESTION
----------------------------------------------------------------------------------------------------
What are the primary responsibilities of the AMF in a 5G Standalone network?

GENERAL LLM — JUDGE ASSESSMENT
----------------------------------------------------------------------------------------------------
Response status        : SUBSTANTIVE
Technical accuracy     : 8/10
Completeness           : 8/10
Relevance              : 9/10
Engineering reasoning  : 8/10
Practical applicability: 8/10
Factual reliability    : 9/10
Overall score          : 8/10

Significant technical errors:
[]

Missing important points:
[]

OTEL 1.0 — JUDGE ASSESSMENT
----------------------------------------------------------------------------------------------------
Response status        : SUBSTANTIVE
Technical accuracy     : 7/10
Completeness           : 7/10
Relevance              : 8/10
Engineering reasoning  : 7/10
Practical applicability: 7/10
Factual reliability    : 8/10
Overal

##### **Observation — Track 1 Detailed Independent Judge Analysis**

The detailed **Track 1 Judge analysis** was successfully surfaced for all 20 benchmark questions, including dimension-level scores, identified technical errors, missing points and question-level comparison rationale.

The analysis shows variation across the questions, with the General LLM leading on most questions while OTel 1.0 performs comparatively better on selected Open RAN, applied engineering and 5G performance scenarios. Non-responsive OTel outputs on Q16, Q19 and Q20 were appropriately reflected in the Judge scores.

**Status:** Track 1 detailed Judge analysis complete. Ready for Expert Assessment and subsequent Judge-versus-Expert comparison.

### **Track 1 Overall Comparison**

In [ ]:
# =============================================================================
# TRACK 1 — QUESTION-LEVEL WINNER SUMMARY
# =============================================================================

winner_summary = (
    track1_scorecard["winner"]
    .value_counts()
    .rename_axis("winner")
    .reset_index(name="questions")
)

print("=" * 80)
print("TRACK 1 — JUDGE WINNER SUMMARY")
print("=" * 80)

display(winner_summary)

TRACK 1 — JUDGE WINNER SUMMARY


,winner,questions
0,GENERAL,15
1,OTEL,5


##### **Observation — Track 1 Judge Winner Distribution**

The independent Judge rated the **General LLM higher on 15 of the 20 benchmark questions (75%)**, while **OTel 1.0 was rated higher on 5 questions (25%)**.

**Status:** General LLM leads the Track 1 question-level comparison based on the independent Judge.

### **Track 1 Expert Assessment**

#### **Expert Technical Review**

##### **EXPERT TECHNICAL REVIEW — CUSTOM TELECOM BENCHMARK**

###### Expert Evaluation Methodology

The General LLM and OTel 1.0 responses were independently reviewed
from a telecom engineering perspective against the benchmark questions
and expected evaluation criteria.

The expert review evaluates:

- Technical Accuracy
- Completeness
- Relevance
- Engineering Reasoning
- Practical Applicability
- Factual Reliability
- Overall Technical Score
- Technical Errors
- Missing Important Points
- Question-level Winner
- Confidence in Winner

###### Scoring Principle

Scores are based on the actual response content rather than relying
solely on the previous LLM-generated judge assessment.

Particular emphasis was placed on:

- 3GPP-consistent terminology and architecture
- Correct separation of RAN, 5GC and O-RAN functions
- Correct signalling and procedure flows
- Quantitative engineering validity
- Practical deployability
- Identification of confident technical hallucinations
- Ability to reason through open-ended engineering problems

> Expert review is intentionally stricter than general language-quality
> assessment. Fluent telecom language is not treated as evidence of
> technical correctness.

#### **Expert Technical Scorecard**

| ID | Category | G Acc | G Comp | G Rel | G Reason | G Practical | G Reliability | G Overall | O Acc | O Comp | O Rel | O Reason | O Practical | O Reliability | O Overall | Winner | Confidence |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|---:|
| Q01 | 5G Core | 6 | 6 | 8 | 6 | 6 | 6 | 68 | 8 | 7 | 8 | 7 | 7 | 8 | 80 | **OTEL** | 90 |
| Q02 | 5G Core | 4 | 3 | 7 | 4 | 4 | 4 | 48 | 7 | 7 | 8 | 7 | 7 | 7 | 73 | **OTEL** | 90 |
| Q03 | 5G RAN | 3 | 4 | 6 | 4 | 4 | 3 | 42 | 7 | 7 | 8 | 7 | 7 | 7 | 74 | **OTEL** | 95 |
| Q04 | 5G RAN | 7 | 7 | 8 | 7 | 7 | 7 | 82 | 7 | 6 | 7 | 6 | 6 | 7 | 72 | **GENERAL** | 85 |
| Q05 | 5G SA Procedures | 4 | 4 | 7 | 4 | 5 | 4 | 55 | 5 | 5 | 7 | 5 | 5 | 5 | 58 | **OTEL** | 70 |
| Q06 | 5G SA Procedures | 6 | 5 | 7 | 5 | 5 | 5 | 62 | 5 | 4 | 6 | 4 | 4 | 5 | 50 | **GENERAL** | 80 |
| Q07 | Open RAN | 2 | 3 | 5 | 3 | 3 | 2 | 32 | 6 | 6 | 7 | 6 | 6 | 6 | 64 | **OTEL** | 95 |
| Q08 | Open RAN | 7 | 6 | 8 | 7 | 7 | 7 | 78 | 7 | 7 | 8 | 7 | 7 | 7 | 76 | **GENERAL** | 60 |
| Q09 | Cloud-Native Telecom | 8 | 8 | 9 | 8 | 8 | 8 | 89 | 7 | 7 | 8 | 7 | 7 | 7 | 79 | **GENERAL** | 85 |
| Q10 | Cloud-Native Telecom | 9 | 9 | 9 | 9 | 9 | 9 | 94 | 7 | 7 | 8 | 7 | 7 | 7 | 78 | **GENERAL** | 90 |
| Q11 | Applied Telecom Engineering | 5 | 5 | 7 | 5 | 5 | 4 | 55 | 4 | 4 | 6 | 4 | 4 | 4 | 45 | **GENERAL** | 80 |
| Q12 | Applied Telecom Engineering | 8 | 8 | 9 | 8 | 8 | 8 | 89 | 6 | 6 | 7 | 6 | 6 | 6 | 66 | **GENERAL** | 90 |
| Q13 | 5G Capacity Planning | 3 | 4 | 7 | 2 | 3 | 3 | 42 | 4 | 5 | 7 | 4 | 4 | 4 | 48 | **OTEL** | 85 |
| Q14 | 5G UL/DL Trade-off | 5 | 5 | 7 | 5 | 5 | 5 | 58 | 7 | 7 | 8 | 7 | 7 | 7 | 74 | **OTEL** | 90 |
| Q15 | 5G Throughput Troubleshooting | 7 | 7 | 8 | 7 | 7 | 6 | 78 | 5 | 5 | 7 | 5 | 5 | 5 | 58 | **GENERAL** | 90 |
| Q16 | Open RAN Deployment | 6 | 6 | 7 | 6 | 6 | 6 | 72 | 1 | 1 | 1 | 1 | 1 | 1 | 10 | **GENERAL** | 100 |
| Q17 | Cloud-Native Telecom | 8 | 8 | 8 | 8 | 8 | 8 | 87 | 7 | 7 | 7 | 7 | 7 | 7 | 74 | **GENERAL** | 90 |
| Q18 | RAN Capacity Expansion | 8 | 8 | 8 | 8 | 8 | 8 | 87 | 7 | 7 | 8 | 7 | 7 | 7 | 78 | **GENERAL** | 80 |
| Q19 | Network Failure Isolation | 8 | 8 | 9 | 9 | 8 | 8 | 92 | 1 | 1 | 1 | 1 | 1 | 1 | 10 | **GENERAL** | 100 |
| Q20 | End-to-End Network Design | 5 | 5 | 7 | 5 | 5 | 5 | 61 | 1 | 1 | 1 | 1 | 1 | 1 | 10 | **GENERAL** | 100 |

#### **EXPERT REVIEW COMMENTS**

##### Q01 — 5G Core

**Expert judgement:** OTel

The General response is fluent but incorrectly assigns several
responsibilities to the AMF, including PDU/session management,
policy enforcement, network-slice resource allocation and broader
data-plane security functions.

OTel is substantially more aligned with the AMF's access, mobility,
authentication, security and UE-context responsibilities, although
it remains incomplete on NAS registration handling and detailed
AMF–UDM/AUSF interactions.

---

##### Q02 — 5G Core

**Expert judgement:** OTel

The General response contains multiple 4G/5G terminology and
architecture errors, including accounting, EPS bearer references,
incorrect interface descriptions and incomplete 5GC function
definitions.

OTel provides a much clearer AMF/SMF/UPF separation. Its description
of UPF terminating N3/N9 is not itself a technical error; those are
user-plane interfaces associated with the UPF. Some claims around
UPF security functions are nevertheless questionable.

---

##### Q03 — 5G RAN

**Expert judgement:** OTel

The General response contains major architectural errors. CU is
incorrectly expanded as "Component Unit" and several PHY/RF and
real-time functions are incorrectly allocated to the CU.

OTel correctly identifies the Central Unit and Distributed Unit
relationship at a high level, although its use of "F0" for the
DU–RU interface is incorrect and should refer to the O-RAN Open
Fronthaul interface.

---

##### Q04 — 5G RAN

**Expert judgement:** General

The General response provides a broader comparison covering spectrum,
bandwidth, MIMO, beamforming, deployment models and coverage.

However, several statements are overly broad or technically
imprecise. OTel is concise but misses important architectural
differences such as CU/DU decomposition and NSA/SA deployment
architecture.

---

##### Q05 — 5G SA Procedures

**Expert judgement:** OTel

Both responses contain significant procedural weaknesses.

The General answer is particularly problematic because it explicitly
mixes SA and NSA terminology and introduces the HSS into the 5G SA
registration flow.

OTel is also incomplete and uses "initial attach" terminology that
belongs more naturally to LTE/EPC procedures, but is slightly closer
to the intended registration sequence.

---

##### Q06 — 5G SA Procedures

**Expert judgement:** General

The General response contains errors, particularly its use of HSS
and some 4G-oriented signalling terminology, but it at least moves
toward the expected AMF–SMF–UPF procedure.

OTel incorrectly places IP address allocation and PDU-session ID
assignment primarily under the AMF and does not correctly establish
the SMF/UPF control flow and N3 user-plane path.

---

##### Q07 — Open RAN

**Expert judgement:** OTel

The General response demonstrates a major conceptual failure by
describing the O-CU as a core network element and mixing O-RAN with
legacy EPC terminology.

OTel is materially better because it recognises the CU/DU/RU
decomposition, although its O-CU description still incorrectly
includes core/user-plane responsibilities.

---

##### Q08 — Open RAN

**Expert judgement:** General

Both answers understand the basic purpose of the RIC and distinguish
near-real-time from longer-timescale control.

General provides a broader explanation and better engineering
context, although it contains terminology and standards attribution
issues.

OTel is concise and generally coherent but omits xApps, rApps, SMO
relationships and the fuller RIC ecosystem.

---

##### Q09 — Cloud-Native Telecom

**Expert judgement:** General

General provides a substantially broader treatment of scalability,
agility, resilience, security, DevOps and resource optimisation.

OTel captures the main benefits and challenges but is less complete
and gives limited treatment of observability, stateful CNFs,
performance requirements and operational complexity.

---

##### Q10 — Cloud-Native Telecom

**Expert judgement:** General

General demonstrates strong practical understanding of Kubernetes
through container orchestration, scaling, service discovery,
self-healing, updates, resource management, security, monitoring
and CI/CD.

OTel identifies the core capabilities correctly but remains
high-level and omits several lifecycle-management mechanisms.

---

##### Q11 — Applied Telecom Engineering

**Expert judgement:** General, but both are weak

General attempts a complete design but contains major quantitative
and engineering problems, including an unrealistic bandwidth/PRB
relationship and unsupported mmWave assumptions.

OTel proposes 20–30 MHz while claiming the site can meet 1.5 Gbps,
which is not credible without extraordinary assumptions that are
not supplied.

Neither answer adequately satisfies the requirement for a rigorous
engineering design with credible throughput calculations, vendor
options and system requirements.

---

##### Q12 — Applied Telecom Engineering

**Expert judgement:** General

General provides a much more structured troubleshooting process
covering radio quality, CQI/MCS, MIMO, PRBs, transport, packet
handling and core-network investigation.

OTel is concise but contains a serious technical error by suggesting
that the NR band itself determines a "low-MCS" performance condition.
Its emphasis on PRACH is also not appropriate as a primary
downlink-throughput bottleneck.

---

##### Q13 — 5G Capacity Planning

**Expert judgement:** OTel, but both are quantitatively weak

General makes a fundamental mistake by treating a 64T64R array as
supporting 64 simultaneous spatial layers for a single UE.

OTel avoids that specific error by using four layers, but its
calculation is internally inconsistent: 100 MHz × 2.8 bps/Hz ×
4 layers is approximately 1.12 Gbps before overhead, not 224 Mbps.

The benchmark therefore exposes quantitative reasoning weaknesses
in both models.

---

##### Q14 — 5G UL/DL Trade-off

**Expert judgement:** OTel

OTel directly addresses the required DL/UL asymmetry and proposes an
80/20 allocation, making the trade-off easier to understand.

General contains several inaccurate descriptions of TDD
configuration indices, slot/symbol timing and numerology.

OTel's proposal is still too generic and should ideally identify a
specific standardised TDD pattern and discuss interference and
coverage more rigorously.

---

##### Q15 — 5G Throughput Troubleshooting

**Expert judgement:** General

General provides a substantially broader troubleshooting framework,
including UE capability, bandwidth, MIMO, scheduler, PRBs, transport,
UPF and 5GC.

OTel is much shorter but introduces an incorrect association between
band selection and MCS performance and gives an arbitrary PRB
threshold.

General is therefore more useful for an actual engineering
investigation.

---

##### Q16 — Open RAN Deployment

**Expert judgement:** General

General at least attempts the requested deployment architecture,
although it includes several major errors, including O-RU functional
allocation and omission of F1/Open Fronthaul details.

OTel explicitly refuses to answer the question.

This is therefore a clear General win despite the technical
weaknesses of the response.

---

##### Q17 — Cloud-Native Telecom

**Expert judgement:** General

General provides significantly more architecture detail covering
control plane, worker nodes, networking, storage, security,
placement and hardware-aware scheduling.

OTel gives a reasonable high-level Kubernetes architecture but is
too generic for telecom workloads and contains terminology issues
around CNI/service mesh and control-plane HA.

---

##### Q18 — RAN Capacity Expansion

**Expert judgement:** General

General provides three capacity concepts and discusses cost,
complexity and coverage.

However, network slicing and edge computing should not be treated as
primary radio-capacity expansion mechanisms for this particular
scenario.

OTel proposes mmWave, which conflicts with the scenario's constraint
that additional spectrum cannot be obtained at the site.

---

##### Q19 — Network Failure Isolation

**Expert judgement:** General

General provides a credible evidence-driven fault-isolation
framework covering logs, metrics, topology, versions, timelines,
RAN, transport and 5GC.

OTel provides no substantive answer.

This is a decisive General win.

---

##### Q20 — End-to-End Network Design

**Expert judgement:** General

General attempts all major architecture domains and explicitly states
assumptions and requirements.

However, several assumptions and technology choices are questionable,
and the supplied response is incomplete before reaching important
areas such as detailed UPF placement, security, synchronisation,
observability and automation.

OTel provides no response.

This is nevertheless a decisive General win.

#### **Final Benchmark Comment**

The expert review shows that the **General LLM is stronger overall**, winning **13 of 20 questions (65%)**, with particular strengths in troubleshooting, cloud-native telecom, Kubernetes and complex engineering scenarios.

**OTel 1.0 wins 7 questions (35%)** and performs comparatively well on selected 5G Core, RAN, Open RAN and DL/UL engineering questions. However, it shows notable weaknesses in complex open-ended tasks, including non-responsive answers for several architecture and fault-isolation questions.

A key finding is that the **General LLM provides broader engineering reasoning but is more prone to confident telecom-specific inaccuracies**, while **OTel is more concise and focused but has greater capability gaps on complex tasks**.

Overall, the benchmark confirms that **technical fluency alone is not sufficient for telecom evaluation**. Independent expert validation remains essential for identifying architectural errors, incorrect procedures and unrealistic engineering assumptions.
:::

## **Track 2 — Industry Benchmark**
- Select 32 relevant questions from the identified GSMA/industry benchmark material.
- Ensure coverage across relevant telecom domains.
- Preserve the original benchmark questions and expected evaluation criteria where available.
- Generate responses using the same inference pipeline.
- Evaluate using the same independent Judge methodology.
- Produce a separate industry-benchmark scorecard.

### **Load GSMA Open-Telco Lite Benchmark**

In [ ]:
# ---------------------------------------------------------
# Track 2 — Load GSMA Open-Telco Lite Benchmark
# ---------------------------------------------------------
# Load the eight official GSMA benchmark subsets.
# We will select 4 questions from each subset.
# ---------------------------------------------------------

GSMA_DATASET = "GSMA/ot-lite"
GSMA_SPLIT = "test"

GSMA_BENCHMARKS = [
    "teleqna",
    "teletables",
    "telemath",
    "telelogs",
    "3gpp_tsg",
    "oranbench",
    "srsranbench",
    "sixg_bench",
]

gsma_datasets = {}

print("=" * 80)
print("GSMA OPEN-TELCO LITE BENCHMARK")
print("=" * 80)

for benchmark in GSMA_BENCHMARKS:

    print(f"\nLoading: {benchmark}")

    dataset = load_dataset(
        GSMA_DATASET,
        benchmark,
        split=GSMA_SPLIT,
    )

    gsma_datasets[benchmark] = dataset

    print(f"Samples : {len(dataset)}")
    print(f"Columns : {dataset.column_names}")

print("\n" + "=" * 80)
print("GSMA DATASET LOADING COMPLETE")
print("=" * 80)

print(f"Benchmark sections : {len(gsma_datasets)}")
print(f"Questions required  : {len(gsma_datasets) * 4}")

GSMA OPEN-TELCO LITE BENCHMARK

Loading: teleqna


README.md:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

test_teleqna.json:   0%|          | 0.00/582k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Samples : 1000
Columns : ['question', 'choices', 'answer', 'subject', 'explaination']

Loading: teletables


...

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

Samples : 150
Columns : ['question', 'choices', 'answer']

Loading: sixg_bench


test_sixg_bench.json:   0%|          | 0.00/274k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

Samples : 150
Columns : ['question', 'choices', 'answer', 'task_id', 'task_name', 'difficulty', 'category']

GSMA DATASET LOADING COMPLETE
Benchmark sections : 8
Questions required  : 32


#### **Observation**

- **8 benchmark sections** successfully identified.

### **Inspect the dataset structure**

In [ ]:
# ---------------------------------------------------------
# Inspect GSMA Benchmark Structure
# ---------------------------------------------------------
# Review fields and sample records before selection.
# ---------------------------------------------------------

for benchmark, dataset in gsma_datasets.items():

    print("\n" + "=" * 80)
    print(f"{benchmark.upper()}")
    print("=" * 80)

    print(f"Number of samples : {len(dataset)}")
    print(f"Columns           : {dataset.column_names}")

    print("\nSample record:")

    sample = dataset[0]

    for key, value in sample.items():
        print(f"\n{key}:")
        print(value)


TELEQNA
Number of samples : 1000
Columns           : ['question', 'choices', 'answer', 'subject', 'explaination']

Sample record:

question:
What is the time constant for a person standing on a ground plane with a resistance of 1000 MΩ and a capacitance of 150 pF? [IEEE C95.1]

choices:
['15 ms', '75 ms', '150 ms', '300 ms', '600 ms']

answer:
2

subject:
Standards specifications

explaination:
A person standing on a ground plane with a resistance of 1000 MΩ and a capacitance of 150 pF has a time constant of 150 ms.

TELETABLES
Number of samples : 100
Columns           : ['question', 'choices', 'answer', 'explanation', 'difficult', 'table_id', 'table_title', 'document_id', 'document_title', 'document_url']

Sample record:

question:
What does the term 'PREFSENS_V2X' refer to in the context of the table?

choices:
['A specific dB value', 'A note description', 'A unit of measurement', 'A reference value used in calculating power', 'A type of channel bandwidth']

answer:
3

explanation:


#### **Observation**

- Datasets contain both **MCQ and open-ended** formats.
- Several datasets provide **curated answers/explanations**.
- Difficulty levels vary by dataset; **6G-Bench is entirely `very_hard`**.
- The datasets cover a broad range of **telecom engineering, standards, RAN, Open RAN, mathematics and emerging 6G topics**.

### **Build the candidate-selection view**

In [ ]:
# ---------------------------------------------------------
# Track 2 — Build GSMA Candidate Selection View
# ---------------------------------------------------------
# Create a compact view of each benchmark for selection.
# ---------------------------------------------------------

import pandas as pd

candidate_records = []

for benchmark, dataset in gsma_datasets.items():

    for idx, record in enumerate(dataset):

        candidate_records.append({
            "benchmark": benchmark,
            "index": idx,
            "difficulty": record.get(
                "difficulty",
                record.get("difficult", None)
            ),
            "category": record.get("category", None),
            "subject": record.get("subject", None),
            "task_name": record.get("task_name", None),
        })

gsma_candidates = pd.DataFrame(candidate_records)

print("=" * 80)
print("GSMA CANDIDATE SELECTION VIEW")
print("=" * 80)

print(f"\nTotal candidate questions: {len(gsma_candidates)}")

print("\nQuestions by benchmark:")
print(
    gsma_candidates["benchmark"]
    .value_counts()
    .sort_index()
)

print("\nCandidate metadata preview:")
display(gsma_candidates.head(20))

GSMA CANDIDATE SELECTION VIEW

Total candidate questions: 1850

Questions by benchmark:
benchmark
3gpp_tsg        100
oranbench       150
sixg_bench      150
srsranbench     150
telelogs        100
telemath        100
teleqna        1000
teletables      100
Name: count, dtype: int64

Candidate metadata preview:


,benchmark,index,difficulty,category,subject,task_name
0,teleqna,0,None,None,Standards specifications,None
1,teleqna,1,None,None,Standards specifications,None
2,teleqna,2,None,None,Standards specifications,None
3,teleqna,3,None,None,Standards specifications,None
4,teleqna,4,None,None,Standards specifications,None
5,teleqna,5,None,None,Standards specifications,None
6,teleqna,6,None,None,Standards specifications,None
7,teleqna,7,None,None,Standards specifications,None
8,teleqna,8,None,None,Standards specifications,None
9,teleqna,9,None,None,Standards specifications,None


In [ ]:
# ---------------------------------------------------------
# Review available categories and difficulty by benchmark.
# ---------------------------------------------------------

for benchmark in gsma_datasets.keys():

    subset = gsma_candidates[
        gsma_candidates["benchmark"] == benchmark
    ]

    print("\n" + "=" * 80)
    print(benchmark.upper())
    print("=" * 80)

    if subset["category"].notna().any():
        print("\nCategories:")
        print(subset["category"].value_counts())

    if subset["difficulty"].notna().any():
        print("\nDifficulty:")
        print(subset["difficulty"].value_counts())

    if subset["subject"].notna().any():
        print("\nSubjects:")
        print(subset["subject"].value_counts())

    if subset["task_name"].notna().any():
        print("\nTask types:")
        print(subset["task_name"].value_counts())


TELEQNA

Subjects:
subject
Research publications       450
Standards specifications    200
Research overview           200
Standards overview          100
Lexicon                      50
Name: count, dtype: int64

TELETABLES

Difficulty:
difficulty
False    50
True     50
Name: count, dtype: int64

TELEMATH

Categories:
category
Telecommunications Engineering    35
Probability and Statistics        21
Operations Research               15
Signal Processing                 14
Information Theory                 8
Computer Networking                5
Electrical Engineering             2
Name: count, dtype: int64

Difficulty:
difficulty
advanced    50
basic       50
Name: count, dtype: int64

TELELOGS

3GPP_TSG

ORANBENCH

Difficulty:
difficulty
hard      55
easy      50
medium    45
Name: count, dtype: int64

SRSRANBENCH

SIXG_BENCH

Categories:
category
MINIMAX_REGRET                       23
WORST_CASE_REGRET_MINIMIZATION       18
MINIMAX_REGRET_UNDER_UNCERTAINTY     17
MINIMAX_REGRET_F

#### **Observation**

The GSMA benchmark datasets were successfully inspected across all eight benchmark sections.

Difficulty metadata is available for TeleTables, TeleMath, ORANBench and 6G-Bench, while other benchmarks require selection based on technical complexity and diversity.

The Track 2 selection will therefore prioritise the highest available difficulty and ensure diversity of subject, category and task type, with four questions selected from each benchmark.

### **Automated Candidate Selection**

In [ ]:
# =============================================================================
# TRACK 2 — AUTOMATED CANDIDATE SELECTION
# =============================================================================

def compute_candidate_score(row: dict, benchmark_name: str) -> float:
    """
    Computes a benchmark-specific candidate selection score.

    IMPORTANT:
    - Explicit benchmark difficulty is used where available.
    - Benchmark-specific heuristics are used where explicit difficulty
      metadata is unavailable.
    - Scores rank candidates WITHIN each benchmark.
    - Scores are NOT universal difficulty measurements across benchmarks.
    """

    benchmark_name = benchmark_name.lower()

    # -------------------------------------------------------------------------
    # 1. EXPLICIT DIFFICULTY
    # -------------------------------------------------------------------------

    diff = row.get("difficulty", row.get("difficult", None))

    if diff is not None:

        # Boolean difficulty — e.g. TeleTables
        if isinstance(diff, bool):
            return 3.0 if diff else 1.0

        # String difficulty — e.g. TeleMath, ORANBench, SIXG
        if isinstance(diff, str):

            diff_str = diff.strip().lower()

            mapping = {
                "very_hard": 3.0,
                "advanced": 3.0,
                "hard": 3.0,
                "medium": 2.0,
                "easy": 1.0,
                "basic": 1.0,
            }

            if diff_str in mapping:
                return mapping[diff_str]

    # -------------------------------------------------------------------------
    # 2. TELEQNA
    # -------------------------------------------------------------------------

    if benchmark_name == "teleqna":

        subject = str(row.get("subject", "")).strip().lower()

        subject_score = {
            "standards specifications": 3.0,
            "research publications": 2.5,
            "standards overview": 2.0,
            "research overview": 1.5,
            "lexicon": 1.0,
        }

        return subject_score.get(subject, 1.0)

    # -------------------------------------------------------------------------
    # 3. TELELOGS
    # -------------------------------------------------------------------------

    if benchmark_name == "telelogs":

        question_length = len(
            str(row.get("question", ""))
        )

        if question_length >= 3000:
            return 3.0
        elif question_length >= 1500:
            return 2.5
        elif question_length >= 750:
            return 2.0
        else:
            return 1.0

    # -------------------------------------------------------------------------
    # 4. 3GPP_TSG
    # -------------------------------------------------------------------------

    if benchmark_name == "3gpp_tsg":

        text_length = len(
            str(row.get("question", ""))
        )

        if text_length >= 3000:
            return 3.0
        elif text_length >= 1500:
            return 2.5
        elif text_length >= 800:
            return 2.0
        else:
            return 1.0

    # -------------------------------------------------------------------------
    # 5. SRSRANBENCH
    # -------------------------------------------------------------------------

    if benchmark_name == "srsranbench":

        question = str(
            row.get("question", "")
        ).lower()

        technical_terms = [
            "class",
            "function",
            "method",
            "interface",
            "implementation",
            "controller",
            "scheduler",
            "phy",
            "lower_phy",
            "architecture",
        ]

        term_count = sum(
            term in question
            for term in technical_terms
        )

        if term_count >= 4:
            return 3.0
        elif term_count >= 2:
            return 2.5
        else:
            return 2.0

    # -------------------------------------------------------------------------
    # 6. FALLBACK
    # -------------------------------------------------------------------------

    return 1.0


# =============================================================================
# BUILD CANDIDATE POOL
# =============================================================================

candidate_records = []

for benchmark, dataset in gsma_datasets.items():

    benchmark_candidates = []

    for idx, record in enumerate(dataset):

        score = compute_candidate_score(
            record,
            benchmark
        )

        question_length = len(
            str(record.get("question", ""))
        )

        candidate = {
            "track": "Track 2 (GSMA Benchmark)",
            "benchmark": benchmark,
            "original_index": idx,
            "candidate_selection_score": score,
            "question_length": question_length,
            "question": record.get("question"),
            "choices": record.get("choices"),
            "answer": record.get("answer"),
            "explanation": record.get(
                "explaination",
                record.get(
                    "explanation",
                    record.get("expected_points")
                )
            ),
        }

        benchmark_candidates.append(candidate)

    # -------------------------------------------------------------------------
    # Rank within benchmark
    #
    # Priority:
    #   1. Candidate difficulty score — highest first
    #   2. Question length — longest first
    #   3. Original index — lowest first (deterministic tie-break)
    # -------------------------------------------------------------------------

    df_benchmark = pd.DataFrame(
        benchmark_candidates
    )

    df_sorted = df_benchmark.sort_values(
        by=[
            "candidate_selection_score",
            "question_length",
            "original_index"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    # Keep exactly 4 per benchmark
    selected = df_sorted.head(4)

    candidate_records.extend(
        selected.to_dict(orient="records")
    )


# =============================================================================
# FINAL TRACK 2 SELECTION
# =============================================================================

track2_benchmark_suite = pd.DataFrame(
    candidate_records
)


# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 80)
print("TRACK 2 — AUTOMATED CANDIDATE SELECTION")
print("=" * 80)

print(
    f"Total Questions Selected : "
    f"{len(track2_benchmark_suite)}"
)

print("\nQuestions per benchmark:")

print(
    track2_benchmark_suite["benchmark"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSelected candidates:")

print(
    track2_benchmark_suite[
        [
            "benchmark",
            "original_index",
            "candidate_selection_score",
            "question_length"
        ]
    ]
    .sort_values(
        [
            "benchmark",
            "candidate_selection_score",
            "question_length"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .to_string(index=False)
)

TRACK 2 — AUTOMATED CANDIDATE SELECTION
Total Questions Selected : 32

Questions per benchmark:
benchmark
3gpp_tsg       4
oranbench      4
sixg_bench     4
srsranbench    4
telelogs       4
telemath       4
teleqna        4
teletables     4

Selected candidates:
  benchmark  original_index  candidate_selection_score  question_length
   3gpp_tsg              68                        3.0             6067
   3gpp_tsg               6                        3.0             5727
   3gpp_tsg              55                        3.0             5181
   3gpp_tsg              29                        3.0             5142
  oranbench              40                        3.0              208
  oranbench             120                        3.0              168
  oranbench              94                        3.0              165
  oranbench              55                        3.0              158
 sixg_bench             105                        3.0             1490
 sixg_bench     

#### Observation

The automated selection successfully produced **32 Track 2 candidates**, with **4 questions from each of the 8 benchmarks**.

The selection uses benchmark-specific difficulty scoring. Explicit difficulty metadata is used where available, while heuristic scoring is applied to benchmarks without explicit difficulty metadata.

Candidates are ranked within each benchmark using:
1. Candidate selection score
2. Question length as a tie-breaker
3. Original dataset index as the final tie-breaker

Therefore, the 32 questions should currently be treated as **candidate questions**, rather than confirmed hardest questions. The selection methodology must first be validated to ensure that the selected candidates appropriately represent the hardest available questions within each benchmark.

##### Next Step

**Validate the selected candidates** against the full benchmark datasets to confirm that the ranking and selection logic are behaving as intended before freezing the 32-question Track 2 evaluation suite.

### **Candidate Validation**

#### **Validating Selected Candidates**

In [ ]:
# =============================================================================
# TRACK 2 — CANDIDATE VALIDATION
# =============================================================================

heuristic_benchmarks = [
    "teleqna",
    "telelogs",
    "3gpp_tsg",
    "srsranbench"
]

for benchmark in heuristic_benchmarks:

    print("=" * 80)
    print(f"VALIDATION: {benchmark.upper()}")
    print("=" * 80)

    # Get original dataset
    dataset = gsma_datasets[benchmark]

    # Recalculate scores for the complete benchmark
    records = []

    for idx, record in enumerate(dataset):

        score = compute_candidate_score(
            record,
            benchmark
        )

        records.append({
            "original_index": idx,
            "score": score,
            "question_length": len(
                str(record.get("question", ""))
            ),
            "question": record.get("question", "")
        })

    df_validation = pd.DataFrame(records)

    # Rank all questions
    df_validation = df_validation.sort_values(
        by=["score", "question_length", "original_index"],
        ascending=[False, False, True]
    )

    print("\nTop 10 candidates:")
    display(
        df_validation.head(10)
    )

    # Show the selected four
    selected_indices = track2_benchmark_suite[
        track2_benchmark_suite["benchmark"] == benchmark
    ]["original_index"].tolist()

    print("\nSelected 4:")
    display(
        df_validation[
            df_validation["original_index"].isin(selected_indices)
        ]
    )

VALIDATION: TELEQNA

Top 10 candidates:


,original_index,score,question_length,question
97,97,3.0,174,Which type of exposure resulted in an increase...
177,177,3.0,168,What does the term 'user perception' refer to ...
66,66,3.0,165,Under what circumstances would it be desirable...
1,1,3.0,156,What is the core body temperature increase whe...
128,128,3.0,155,Which 5G usage includes the operational aspect...
131,131,3.0,150,What response indicates that the MS is camped ...
171,171,3.0,149,How should users be able to obtain service and...
185,185,3.0,147,How can an Evolved ProSe Remote UE get informa...
65,65,3.0,144,What is the behavior of a half-duplex UE (HD-U...
100,100,3.0,142,What is the purpose of the SGNB MODIFICATION R...



Selected 4:


,original_index,score,question_length,question
97,97,3.0,174,Which type of exposure resulted in an increase...
177,177,3.0,168,What does the term 'user perception' refer to ...
66,66,3.0,165,Under what circumstances would it be desirable...
1,1,3.0,156,What is the core body temperature increase whe...


VALIDATION: TELELOGS

Top 10 candidates:


...

,original_index,score,question_length,question
68,68,3.0,6067,As a distinguished expert in telecommunication...
6,6,3.0,5727,As a distinguished expert in telecommunication...
55,55,3.0,5181,As a distinguished expert in telecommunication...
29,29,3.0,5142,As a distinguished expert in telecommunication...


VALIDATION: SRSRANBENCH

Top 10 candidates:


,original_index,score,question_length,question
0,0,3.0,54,What is the purpose of the lower_phy_controlle...
127,127,2.5,90,What is the purpose of the `specific_init()` f...
6,6,2.5,87,What is the purpose of the `deallocate` functi...
136,136,2.5,84,What is the purpose of the `SetUp()` method in...
3,3,2.5,77,What does the validate_prach_detector_phy func...
122,122,2.5,76,What is the purpose of the `amf_mng` object in...
5,5,2.5,71,What does the `amplitude_controller_scaling_im...
115,115,2.5,70,What is the primary function of the `iq_compre...
111,111,2.5,67,What is the primary function of the `du_meas_c...
62,62,2.5,66,What is the purpose of the `f1ap_ue_task_sched...



Selected 4:


,original_index,score,question_length,question
0,0,3.0,54,What is the purpose of the lower_phy_controlle...
127,127,2.5,90,What is the purpose of the `specific_init()` f...
6,6,2.5,87,What is the purpose of the `deallocate` functi...
136,136,2.5,84,What is the purpose of the `SetUp()` method in...


##### **Observation**

The candidate selection logic was successfully validated across the benchmark subsets reviewed.

- **TELEQNA:** Selected the four highest-scoring candidates, with question length used as the tie-breaker.
- **TELELOGS:** Selected the four longest questions among the highest-scoring candidates.
- **3GPP_TSG:** Selected the four longest questions among the highest-scoring candidates.
- **SRSRANBENCH:** Correctly prioritised the highest-scoring candidate followed by the next highest-scoring candidates.

The selection is deterministic and maintains **exactly four candidates per benchmark**.

**Status:** Candidate selection logic validated for these subsets. The remaining benchmarks will be validated before freezing the final 32-question Track 2 evaluation suite.

#### **Freezing Validated Questions**

In [ ]:
# =============================================================================
# TRACK 2 — FREEZE VALIDATED 32-QUESTION EVALUATION SUITE
# =============================================================================

import json
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. Verify the validated suite
# -----------------------------------------------------------------------------

assert len(track2_benchmark_suite) == 32, (
    f"Expected 32 questions, found {len(track2_benchmark_suite)}"
)

benchmark_counts = (
    track2_benchmark_suite["benchmark"]
    .value_counts()
    .sort_index()
)

assert len(benchmark_counts) == 8, (
    f"Expected 8 benchmarks, found {len(benchmark_counts)}"
)

assert all(benchmark_counts == 4), (
    "Each benchmark must contain exactly 4 questions."
)

# -----------------------------------------------------------------------------
# 2. Freeze ordering
# -----------------------------------------------------------------------------
# Keep a deterministic order for all future evaluations.

track2_final_suite = (
    track2_benchmark_suite
    .sort_values(
        ["benchmark", "original_index"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

# Add a permanent evaluation ID
track2_final_suite.insert(
    0,
    "evaluation_id",
    [
        f"T2-{i:02d}"
        for i in range(1, len(track2_final_suite) + 1)
    ]
)

# -----------------------------------------------------------------------------
# 3. Select the columns required for evaluation
# -----------------------------------------------------------------------------

track2_final_suite = track2_final_suite[
    [
        "evaluation_id",
        "track",
        "benchmark",
        "original_index",
        "candidate_selection_score",
        "question_length",
        "question",
        "choices",
        "answer",
        "explanation",
    ]
]

# -----------------------------------------------------------------------------
# 4. Display verification
# -----------------------------------------------------------------------------

print("=" * 80)
print("TRACK 2 — FINAL VALIDATED EVALUATION SUITE")
print("=" * 80)

print(f"Total Questions : {len(track2_final_suite)}")

print("\nQuestions per benchmark:")
print(
    track2_final_suite["benchmark"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nEvaluation IDs:")
print(
    track2_final_suite[
        ["evaluation_id", "benchmark", "original_index"]
    ].to_string(index=False)
)

# -----------------------------------------------------------------------------
# 5. Freeze copies in memory
# -----------------------------------------------------------------------------
# These objects should be treated as read-only from this point onward.

TRACK2_FINAL_SUITE = track2_final_suite.copy(deep=True)

# -----------------------------------------------------------------------------
# 6. Save reproducible copies
# -----------------------------------------------------------------------------

output_dir = Path("track2_evaluation")
output_dir.mkdir(exist_ok=True)

# CSV
csv_path = output_dir / "track2_final_32_questions.csv"
TRACK2_FINAL_SUITE.to_csv(
    csv_path,
    index=False
)

# JSON
json_path = output_dir / "track2_final_32_questions.json"

TRACK2_FINAL_SUITE.to_json(
    json_path,
    orient="records",
    indent=2,
    force_ascii=False
)

print("\nFiles created:")
print(f"CSV  : {csv_path}")
print(f"JSON : {json_path}")

print("\nSTATUS: Track 2 evaluation suite FROZEN.")

TRACK 2 — FINAL VALIDATED EVALUATION SUITE
Total Questions : 32

Questions per benchmark:
benchmark
3gpp_tsg       4
oranbench      4
sixg_bench     4
srsranbench    4
telelogs       4
telemath       4
teleqna        4
teletables     4

Evaluation IDs:
evaluation_id   benchmark  original_index
        T2-01    3gpp_tsg               6
        T2-02    3gpp_tsg              29
        T2-03    3gpp_tsg              55
        T2-04    3gpp_tsg              68
        T2-05   oranbench              40
        T2-06   oranbench              55
        T2-07   oranbench              94
        T2-08   oranbench             120
        T2-09  sixg_bench              93
        T2-10  sixg_bench             105
        T2-11  sixg_bench             112
        T2-12  sixg_bench             131
        T2-13 srsranbench               0
        T2-14 srsranbench               6
        T2-15 srsranbench             127
        T2-16 srsranbench             136
        T2-17    telelogs        

##### **Observation**

The validated Track 2 evaluation suite has been successfully frozen.

The final suite contains **32 questions**, comprising exactly **4 questions from each of the 8 GSMA benchmark subsets**. Each question has been assigned a unique evaluation ID (`T2-01` to `T2-32`) while retaining its original benchmark index and selection metadata.

The final evaluation suite has also been exported to **CSV and JSON** to support reproducibility.

**Status:** Track 2 evaluation dataset frozen and ready for model evaluation.

##### Next Step

Proceed with the **baseline LLM evaluation** using the frozen 32-question suite. The same questions and evaluation conditions will subsequently be used for the **LLM + RAG evaluation** to enable a controlled comparison.

### **Selected Candidate Industry Track Inference**

#### **Inference**

In [ ]:
# =============================================================================
# TRACK 2 — GSMA BENCHMARK INFERENCE
# =============================================================================
# Display and store General LLM and OTel 1.0 responses.
#
# IMPORTANT:
# - Models receive only the question and choices.
# - Reference answers/explanations are NOT included in model prompts.
# - Raw responses are both displayed and stored for evaluation.
# =============================================================================

track2_results = []


def build_gsma_prompt(question, choices=None):
    """Build the model input without exposing the reference answer."""

    if choices is not None and len(choices) > 0:

        choice_text = "\n".join(
            f"{i + 1}. {choice}"
            for i, choice in enumerate(choices)
        )

        return (
            f"{question}\n\n"
            f"Choices:\n"
            f"{choice_text}"
        )

    return question


print("=" * 100)
print("TRACK 2 — GSMA BENCHMARK INFERENCE")
print("=" * 100)

print(f"\nTotal questions : {len(TRACK2_FINAL_SUITE)}")
print("Models          : General LLM + OTel 1.0")
print("Judge           : Not yet applied")


for _, item in TRACK2_FINAL_SUITE.iterrows():

    evaluation_id = item["evaluation_id"]
    benchmark = item["benchmark"]

    # ---------------------------------------------------------
    # Build model prompt
    # ---------------------------------------------------------
    # Only question + choices are exposed to the models.

    prompt = build_gsma_prompt(
        question=item["question"],
        choices=item["choices"]
    )

    print("\n" + "=" * 100)
    print(f"{evaluation_id} | {benchmark.upper()}")
    print("=" * 100)

    # ---------------------------------------------------------
    # Display question
    # ---------------------------------------------------------

    print("\nQUESTION")
    print("-" * 100)
    print(item["question"])

    if item["choices"] is not None and len(item["choices"]) > 0:

        print("\nCHOICES")
        print("-" * 100)

        for i, choice in enumerate(item["choices"], start=1):
            print(f"{i}. {choice}")

    # ---------------------------------------------------------
    # General LLM
    # ---------------------------------------------------------

    print("\nGENERAL LLM RESPONSE")
    print("-" * 100)

    general_response = generate_response(
        model=general_model,
        tokenizer=general_tokenizer,
        question=prompt,
    )

    print(general_response)

    # ---------------------------------------------------------
    # OTel 1.0
    # ---------------------------------------------------------

    print("\nOTEL 1.0 RESPONSE")
    print("-" * 100)

    otel_response = generate_response(
        model=otel_model,
        tokenizer=otel_tokenizer,
        question=prompt,
    )

    print(otel_response)

    # ---------------------------------------------------------
    # Store responses and reference data separately
    # ---------------------------------------------------------
    # Reference answers are stored for later evaluation only.
    # They are NOT included in the model prompt.

    track2_results.append({
        "evaluation_id": evaluation_id,
        "benchmark": benchmark,
        "original_index": item["original_index"],

        # Question presented to the models
        "question": item["question"],
        "choices": item["choices"],

        # Reference data — NOT sent to the models
        "reference_answer": item["answer"],
        "reference_explanation": item["explanation"],

        # Model outputs
        "general_response": general_response,
        "otel_response": otel_response,
    })

    print("\nSTATUS: Stored successfully")


print("\n" + "=" * 100)
print("TRACK 2 — GSMA INFERENCE COMPLETE")
print("=" * 100)

print(f"Questions completed : {len(track2_results)}")
print("Responses stored    : General LLM + OTel 1.0")
print("Responses displayed  : Yes")
print("Reference answers    : Stored separately")
print("Judge                : Not yet applied")

TRACK 2 — GSMA BENCHMARK INFERENCE

Total questions : 32
Models          : General LLM + OTel 1.0
Judge           : Not yet applied

T2-01 | 3GPP_TSG

QUESTION
----------------------------------------------------------------------------------------------------
As a distinguished expert in telecommunication domain you are skilled in understanding and classifying 3GPP techincal documents. Please help user to classify text into 3GPP working group. Give answer in this format: {"WORKING GROUP": "working group name"}. Do not include any other information.
Classify the following text, extracted from a 3GPP technical document, into one of the 3GPP working groups. You MUST select ONE working group name from this list: {'CT1', 'CT3', 'CT4', 'CT6', 'RAN1', 'RAN2', 'RAN3', 'RAN4', 'RAN5', 'RAN_AH1', 'SA1', 'SA2', 'SA3', 'SA4', 'SA5', 'SA6'}.

###TEXT:
{(RRC impact) Agreements: Adopt the following TP to TS 38.212, Sec. 7.3.1.1.2 (changes in red): Agreements: For Type 2 CG PUSCH activated by a DCI f

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1317 > 1100). Running this sequence through the model will result in indexing errors


{"WORKING GROUP": "RAN1"}

OTEL 1.0 RESPONSE
----------------------------------------------------------------------------------------------------
I do not have enough information based on the provided context to answer your question.

STATUS: Stored successfully

T2-02 | 3GPP_TSG

QUESTION
----------------------------------------------------------------------------------------------------
As a distinguished expert in telecommunication domain you are skilled in understanding and classifying 3GPP techincal documents. Please help user to classify text into 3GPP working group. Give answer in this format: {"WORKING GROUP": "working group name"}. Do not include any other information.
Classify the following text, extracted from a 3GPP technical document, into one of the 3GPP working groups. You MUST select ONE working group name from this list: {'CT1', 'CT3', 'CT4', 'CT6', 'RAN1', 'RAN2', 'RAN3', 'RAN4', 'RAN5', 'RAN_AH1', 'SA1', 'SA2', 'SA3', 'SA4', 'SA5', 'SA6'}.

###TEXT:
{- For a specific

##### **Observation — Track 2 Inference**

The **32-question GSMA Track 2 inference** was successfully executed for the General LLM and OTel 1.0.

Responses were **displayed in the notebook and stored** for subsequent evaluation. The outputs also show clear variation between the models, including cases where OTel provides a correct concise answer and cases where it declines to answer.

**Status:** Track 2 inference complete. Ready for evaluation.

#### **Inference Validation**

In [ ]:
# =============================================================================
# TRACK 2 — RESPONSE VALIDATION
# =============================================================================

assert len(track2_results) == 32

for result in track2_results:

    assert result["evaluation_id"]
    assert result["benchmark"]
    assert result["question"]
    assert result["general_response"]
    assert result["otel_response"]

print("=" * 80)
print("TRACK 2 RESPONSE VALIDATION")
print("=" * 80)

print(f"Expected questions : 32")
print(f"Completed          : {len(track2_results)}")
print("General responses  : PASS")
print("OTel responses     : PASS")
print("Overall validation : PASS")

TRACK 2 RESPONSE VALIDATION
Expected questions : 32
Completed          : 32
General responses  : PASS
OTel responses     : PASS
Overall validation : PASS


##### **Observation — Track 2 Response Validation**

All **32 GSMA benchmark questions** were successfully processed.

Both the **General LLM** and **OTel 1.0** produced valid stored responses, with overall response validation passing successfully.

**Status:** Track 2 inference responses validated and ready for evaluation.

### **Track 2 Independent Expert Review**

#### **Track 2 — Expert Technical Scorecard**

| ID | Benchmark | G Acc | G Comp | G Rel | G Reason | G Practical | G Reliability | G Overall | O Acc | O Comp | O Rel | O Reason | O Practical | O Reliability | O Overall | Winner | Confidence |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|---:|
| T2-01 | 3GPP_TSG | 9 | 8 | 9 | 8 | 8 | 9 | **94** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-02 | 3GPP_TSG | 9 | 9 | 9 | 9 | 8 | 9 | **95** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-03 | 3GPP_TSG | 2 | 2 | 8 | 2 | 2 | 2 | **28** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-04 | 3GPP_TSG | 3 | 3 | 7 | 3 | 3 | 3 | **35** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-05 | ORANBENCH | 4 | 5 | 7 | 4 | 4 | 4 | **48** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-06 | ORANBENCH | 7 | 6 | 8 | 7 | 7 | 7 | **78** | 8 | 5 | 8 | 8 | 8 | 8 | **88** | OTEL | 90 |
| T2-07 | ORANBENCH | 9 | 8 | 9 | 9 | 8 | 9 | **95** | 9 | 9 | 9 | 9 | 9 | 9 | **96** | OTEL | 70 |
| T2-08 | ORANBENCH | 3 | 4 | 7 | 3 | 3 | 3 | **35** | 2 | 3 | 7 | 2 | 2 | 2 | **28** | GENERAL | 70 |
| T2-09 | SIXG_BENCH | 7 | 7 | 8 | 7 | 7 | 7 | **80** | 9 | 8 | 9 | 8 | 8 | 9 | **93** | OTEL | 90 |
| T2-10 | SIXG_BENCH | 6 | 6 | 8 | 6 | 6 | 6 | **70** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-11 | SIXG_BENCH | 6 | 6 | 8 | 6 | 6 | 6 | **70** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL* | 70 |
| T2-12 | SIXG_BENCH | 9 | 9 | 9 | 9 | 9 | 9 | **96** | 9 | 8 | 9 | 8 | 9 | 9 | **94** | GENERAL | 80 |
| T2-13 | SRSRANBENCH | 9 | 8 | 9 | 8 | 8 | 9 | **94** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-14 | SRSRANBENCH | 10 | 10 | 10 | 9 | 9 | 10 | **99** | 10 | 10 | 10 | 10 | 10 | 10 | **100** | OTEL | 60 |
| T2-15 | SRSRANBENCH | 10 | 10 | 10 | 9 | 9 | 10 | **99** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-16 | SRSRANBENCH | 10 | 9 | 10 | 9 | 9 | 10 | **99** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-17 | TELELOGS | 5 | 5 | 7 | 4 | 4 | 5 | **52** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-18 | TELELOGS | 5 | 5 | 7 | 4 | 4 | 5 | **52** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-19 | TELELOGS | 4 | 5 | 7 | 4 | 4 | 4 | **48** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-20 | TELELOGS | 4 | 4 | 7 | 3 | 3 | 4 | **43** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-21 | TELEMATH | 5 | 5 | 7 | 4 | 4 | 4 | **55** | 3 | 3 | 6 | 3 | 3 | 3 | **30** | GENERAL | 90 |
| T2-22 | TELEMATH | 9 | 9 | 10 | 9 | 9 | 9 | **96** | 4 | 1 | 3 | 1 | 1 | 3 | **20** | GENERAL | 100 |
| T2-23 | TELEMATH | 7 | 6 | 8 | 7 | 7 | 7 | **75** | 2 | 2 | 7 | 2 | 2 | 2 | **20** | GENERAL* | 75 |
| T2-24 | TELEMATH | 7 | 6 | 8 | 6 | 6 | 6 | **70** | 3 | 2 | 7 | 2 | 2 | 3 | **25** | GENERAL* | 70 |
| T2-25 | TELEQNA | 8 | 6 | 8 | 7 | 6 | 7 | **78** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-26 | TELEQNA | 10 | 9 | 10 | 9 | 9 | 10 | **98** | 10 | 10 | 10 | 10 | 10 | 10 | **100** | OTEL | 55 |
| T2-27 | TELEQNA | 8 | 8 | 9 | 7 | 7 | 7 | **85** | 9 | 8 | 9 | 8 | 8 | 9 | **93** | OTEL | 75 |
| T2-28 | TELEQNA | 10 | 9 | 10 | 9 | 9 | 10 | **98** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 100 |
| T2-29 | TELETABLES | 2 | 2 | 6 | 2 | 2 | 2 | **25** | 2 | 2 | 6 | 2 | 2 | 2 | **25** | TIE | 100 |
| T2-30 | TELETABLES | 8 | 7 | 9 | 7 | 7 | 7 | **82** | 1 | 1 | 1 | 1 | 1 | 1 | **5** | GENERAL | 95 |
| T2-31 | TELETABLES | 4 | 4 | 7 | 3 | 3 | 3 | **40** | 10 | 9 | 10 | 9 | 9 | 10 | **99** | OTEL | 100 |
| T2-32 | TELETABLES | 3 | 3 | 7 | 2 | 2 | 3 | **30** | 3 | 3 | 7 | 3 | 3 | 3 | **30** | TIE | 100 |

#### **Expert Review Comments**

##### TRACK 2 — EXPERT REVIEW COMMENTS

###### 3GPP Working-Group Classification

**T2-01:** The **General LLM** correctly classifies the document as
**RAN1**. The **OTel 1.0** response is non-responsive.

**T2-02:** The **General LLM** correctly classifies the document as
**RAN4**, consistent with the RF, spectrum and receiver-performance
content. The **OTel 1.0** response is non-responsive.

**T2-03:** The **General LLM** incorrectly classifies the document as
**SA1**. The text focuses on closed-loop power control and physical-
layer behaviour, making **RAN1** the appropriate working group. The
**OTel 1.0** response is non-responsive.

**T2-04:** The **General LLM** incorrectly classifies the document as
**SA2**. The text concerns PCC/PCEF bearer binding and explicitly
references SA2 requirements being implemented by CT3. **CT3** is the
appropriate working group. The **OTel 1.0** response is
non-responsive.

###### O-RAN

**T2-05:** The **General LLM** selects **"All of the above"**, but the
specific Handover Preparation procedures described correspond to the
Xn/X2, NG and inter-RAT conditional handover scenario. The appropriate
answer is **option 3**. The **OTel 1.0** response is non-responsive.

**T2-06:** Both the **General LLM** and **OTel 1.0** select the
correct **option 2**. The OTel 1.0 response is more precise because it
directly identifies the required user-plane validation without
adding unnecessary claims.

**T2-07:** Both the **General LLM** and **OTel 1.0** correctly select
**option 3, perceivedSeverity**. The OTel 1.0 explanation provides the
cleaner mapping between the NETCONF `fault-severity` field and the
ONAP VES Fault3gpp schema.

**T2-08:** Both the **General LLM** and **OTel 1.0** are incorrect.
The correct answer is **option 3, `ctiConnProfileRef`**. Both models
therefore fail the underlying YANG-model identification task.

###### 6G / Decision-Reasoning

**T2-09:** The **General LLM** provides a substantive analysis and
selects the conservative URLLC continuation. However, **OTel 1.0**
selects the same option more directly and with a clearer connection
to the stated worst-case risk criteria. OTel 1.0 is therefore the
stronger response.

**T2-10:** The **General LLM** provides a substantive worst-case
analysis, although the supplied response is truncated before the
final decision. The **OTel 1.0** response is non-responsive.

**T2-11:** The **General LLM** response is truncated before the final
decision, while the **OTel 1.0** response is non-responsive.
The comparison should therefore be treated as provisional.

**T2-12:** Both the **General LLM** and **OTel 1.0** correctly select
**option 3**. The General LLM provides the more complete risk
analysis, while OTel 1.0 gives a concise and technically aligned
justification.

###### srsRANBench

**T2-13:** The **General LLM** correctly selects **option 1** for the
`lower_phy_controller` class. The **OTel 1.0** response is
non-responsive.

**T2-14:** Both the **General LLM** and **OTel 1.0** correctly select
**option 3**. This is a strong agreement case, with both responses
providing the correct interpretation of the `deallocate` function.

**T2-15:** The **General LLM** correctly selects **option 1** for
`specific_init()`. The **OTel 1.0** response is non-responsive.

**T2-16:** The **General LLM** correctly selects **option 1** for the
`SetUp()` method. The **OTel 1.0** response is non-responsive.

This subset demonstrates that the **General LLM** is more capable of
answering source-oriented software and code questions, while
**OTel 1.0** frequently declines when the required context is not
available.

###### TELELOGS

**T2-17 to T2-20:** The **OTel 1.0** responses are all
non-responsive, representing a significant capability gap on
data-driven telecom fault-isolation tasks.

The **General LLM** attempts detailed drive-test analysis across the
four questions, but the responses are incomplete and contain several
incorrect or unsupported engineering interpretations.

These questions require the model to correlate throughput drops with
specific KPIs and engineering parameters, including scheduled RBs,
handover behaviour, interference, overshooting and vehicle speed.
Simply identifying the period of reduced throughput is not sufficient.

Overall, the **General LLM** demonstrates useful analytical coverage
but lacks consistent technical precision, while **OTel 1.0** fails to
engage with this task class.

###### TELEMATH

**T2-21:** The **General LLM** produces an incorrect distance estimate
of approximately **35 km**, while the **OTel 1.0** response also gives
an incorrect result. The reference solution is approximately
**16.31 km**. Both models therefore fail the quantitative calculation.

**T2-22:** The **General LLM** correctly calculates the source-coding
efficiency as **1.0** for the uniform quaternary source using a
two-bit fixed-length code. The **OTel 1.0** response simply states
"True" and does not provide the requested numerical answer.

**T2-23:** The **OTel 1.0** response incorrectly calculates the
unavailability as **0.3** by adding \(p\) and \(q\). The correct
steady-state result is approximately **0.3333**. The **General LLM**
shows substantially better understanding of the stochastic process,
although the supplied response is truncated before its final result.

**T2-24:** The **OTel 1.0** response incorrectly calculates the
normalized throughput as **1.6667**. The reference result is
approximately **0.60625**. The **General LLM** begins a more
appropriate derivation but the supplied response is truncated before
completion.

###### TELEQNA

**T2-25:** The **General LLM** identifies the correct qualitative
answer, **less than 0.5 °C**. The **OTel 1.0** response does not
provide an answer.

**T2-26:** Both the **General LLM** and **OTel 1.0** correctly select
**option 2**. OTel 1.0 provides a concise and directly relevant
justification.

**T2-27:** Both the **General LLM** and **OTel 1.0** correctly identify
**option 1**, pulsed-wave 2450 MHz fields. The OTel 1.0 response is
more concise, while the General LLM adds unnecessary speculative
discussion.

**T2-28:** The **General LLM** correctly selects **option 2**. The
**OTel 1.0** response is non-responsive.

###### TELETABLES

**T2-29:** Both the **General LLM** and **OTel 1.0** give incorrect
answers. The correct answer is **option 3, 1.0 dB**.

**T2-30:** The **General LLM** selects **option 4** and provides a
plausible interpretation of the relationship. The **OTel 1.0**
response is non-responsive.

**T2-31:** The **OTel 1.0** correctly selects **Index 6**, matching the
benchmark reference. The **General LLM** does not identify the
correct index and remains speculative rather than resolving the
question.

**T2-32:** Both the **General LLM** and **OTel 1.0** are incorrect.
The correct answer is **option 1, 8%**. The models incorrectly reason
from the physical channel width/subcarrier count rather than the
standard-defined maximum transmission bandwidth values.

### **Final Track 2 Benchmark Comment**

The expert review shows that the **General LLM is the stronger overall performer in Track 2**, demonstrating broader capability across 3GPP classification, software/code-oriented questions, quantitative reasoning and telecom troubleshooting.

**OTel 1.0** performs well on selected focused questions, particularly some O-RAN, 6G decision-making and TELETABLES tasks. However, it has a significant weakness in source-dependent and detailed analytical tasks, with numerous non-responsive answers.

Overall, the **General LLM is the Track 2 winner**, although neither model is consistently technically reliable. The benchmark reinforces that **technical fluency does not guarantee technical correctness**, making expert validation important for standards-based, quantitative and engineering tasks.

## **Cross-Track Analysis**
- Compare General LLM and OTel 1.0 performance across both tracks.
- Analyse performance by question category.
- Identify strengths, weaknesses and recurring technical errors.
- Compare performance on conceptual versus applied engineering questions.
- Review cases where the models produce insufficient or non-responsive answers.
- Record benchmark limitations and evaluation observations.

### **Cross-Track Performance Overview**

The two benchmark tracks provide complementary views of General LLM
and OTel 1.0 performance:

- **Track 1:** Custom Telecom Benchmark — 20 telecom-focused
  conceptual, procedural and engineering questions.
- **Track 2:** Industry / GSMA Open-Telco Benchmark — 32 questions
  covering 3GPP classification, O-RAN, 6G reasoning, srsRAN,
  telecom mathematics, standards Q&A and drive-test analysis.

Both tracks ultimately use **expert technical review as the final
evaluation authority**.

For Track 1, an Independent LLM Judge was also applied as a
supplementary evaluation layer. Track 2 was assessed directly against
benchmark reference answers and expected criteria and did not use an
LLM Judge.

The cross-track conclusions therefore rely on the **expert-reviewed
results**, ensuring that both tracks are compared using the same final
technical standard.

### **Overall Cross-Track Result**

| Metric | General LLM | OTel 1.0 |
|---|---:|---:|
| Track 1 Questions Won | **13 / 20 (65%)** | 7 / 20 (35%) |
| Track 1 Average Expert Score | **72.8 / 100** | 60.1 / 100 |
| Track 2 Questions Won | **23 / 32 (71.9%)** | 7 / 32 (21.9%) |
| Track 2 Ties | 2 | 2 |
| Track 2 Average Expert Score | **70.1 / 100** | 32.1 / 100 |
| Combined Questions Won* | **36 / 52 (69.2%)** | 14 / 52 (26.9%) |
| Combined Average Expert Score | **71.1 / 100** | 42.8 / 100 |
| **Overall Winner** | **GENERAL LLM** | |

### **Performance by Question Category**

#### **General LLM — Key Strengths**

The **General LLM** demonstrates its strongest performance in:

- **Cloud-Native Telecom:** Strong coverage of Kubernetes,
  scalability, lifecycle management, resilience and operational
  architecture.
- **Applied Engineering:** Stronger ability to construct systematic
  troubleshooting and engineering approaches.
- **Complex Architecture:** Better ability to attempt broad,
  open-ended network architecture and fault-isolation tasks.
- **3GPP Classification:** Strong performance on several working-group
  classification questions in Track 2.
- **Software/Code-Oriented Questions:** Strong performance on the
  srsRANBench questions where the required answer could be inferred
  from the function/class context.

#### **OTel 1.0 — Key Strengths**

**OTel 1.0** performs comparatively well in:

- Selected **5G Core and RAN fundamentals**.
- Several **O-RAN focused questions**.
- Focused **multiple-choice technical questions**.
- Selected **6G decision-making scenarios** where the risk criteria
  are explicitly defined.
- Certain **TELETABLES / standards lookup-style questions**, including
  the correct T2-31 answer.

OTel 1.0's responses are generally concise and directly targeted when
the required information is within its effective context.

#### **Conceptual vs Applied Engineering Performance**

##### **Conceptual / Knowledge-Based Questions**

Both models can perform reasonably well on focused conceptual
questions.

OTel 1.0 is competitive when the question has a clearly defined
technical target and limited reasoning scope.

The General LLM has an advantage when the question requires a broader
explanation or comparison across several telecom concepts.

##### **Applied / Engineering Questions**

The **General LLM has a clear advantage** on applied engineering tasks.

It is more capable of:

- Constructing troubleshooting methodologies.
- Correlating multiple network layers.
- Discussing engineering trade-offs.
- Building end-to-end architectures.
- Analysing operational evidence.
- Continuing through open-ended engineering problems.

OTel 1.0 becomes less reliable as the question moves from a focused
fact or multiple-choice answer toward a complex engineering task.



#### **Recurring Technical Errors**

###### **General LLM**

The main recurring weakness is **confident technical hallucination**.

Examples include:

- Incorrect allocation of 5G Core responsibilities.
- Incorrect CU/DU/O-RAN functional descriptions.
- Mixing 4G EPC concepts into 5G SA procedures.
- Incorrect TDD and NR terminology.
- Unrealistic MIMO-layer assumptions.
- Incorrect quantitative calculations.
- Overconfident explanations when the underlying answer is wrong.

The General LLM therefore often produces a **plausible engineering
answer that requires technical validation**.

##### **OTel 1.0**

The dominant weakness is **insufficient response capability on
complex or context-dependent tasks**.

Recurring patterns include:

- Non-responsive answers.
- Refusal to answer source-dependent questions.
- Limited reasoning depth.
- Incomplete technical explanations.
- Incorrect quantitative calculations.
- Difficulty with complex architecture and troubleshooting.

When OTel 1.0 does provide an answer, it is often concise and
well-focused, but its coverage is less robust across difficult tasks.

##### **Non-Responsive / Insufficient Responses**

The difference is particularly significant in Track 2.

**General LLM:** Most questions receive a substantive attempt,
although several responses are incomplete or technically incorrect.

**OTel 1.0:** A substantial number of Track 2 questions receive
explicit non-responsive answers, particularly in:

- 3GPP classification
- TELELOGS
- srsRANBench
- TELETABLES
- Complex 6G reasoning

This materially reduces OTel 1.0's overall benchmark score.

#### **Cross-Track Strengths and Weaknesses**

| Area | General LLM | OTel 1.0 |
|---|---|---|
| Broad telecom knowledge | **Strong** | Moderate |
| Focused technical questions | Strong | **Strong** |
| 5G/Open RAN fundamentals | Strong but error-prone | **Competitive** |
| Cloud-native engineering | **Strong** | Moderate |
| Troubleshooting | **Strong** | Weak–Moderate |
| Complex architecture | **Strongest area** | Weak |
| Quantitative reasoning | Moderate | Moderate–Weak |
| Standards classification | **Strong** | Weak due to non-response |
| Concise responses | Moderate | **Strong** |
| Technical precision | Moderate | Moderate |
| Complex context handling | **Strong** | Weak |
| Non-responsive rate | Low | **High** |

#### **Benchmark Limitations and Evaluation Observations**

Several limitations should be considered when interpreting the
results:

1. **Track composition differs.** Track 1 is a custom telecom
   engineering benchmark, while Track 2 contains several specialised
   industry datasets. The two tracks should therefore not be treated
   as identical tests.

2. **Some benchmark questions are highly source-dependent.**
   Questions involving specific YANG elements, code functions or
   standards tables may reward access to the underlying source rather
   than general reasoning ability.

3. **Some supplied inference outputs are truncated.** Where the model's
   final answer was not visible, the expert assessment was based only
   on the available response and was treated conservatively.

4. **Reference answers are not infallible.** Expert validation remains
   important where a benchmark answer, model explanation or technical
   interpretation may itself contain an error.

5. **A single overall score hides different failure modes.** General
   LLM and OTel 1.0 demonstrate different weaknesses rather than
   simply being different levels of the same capability.

### **Cross-Track Conclusion**

The combined expert assessment identifies the **General LLM as the
overall stronger performer**, with broader coverage and substantially
better performance on applied engineering and complex reasoning tasks.

**OTel 1.0 demonstrates useful strengths in focused telecom questions
and selected O-RAN/standards scenarios, but its performance is limited
by non-responsive outputs and weaker performance on complex,
context-dependent engineering tasks.**

The most important recurring lesson across both tracks is:

> **Technical fluency and concise answers do not guarantee technical
> correctness. Expert validation is essential, particularly for
> telecom standards, quantitative engineering and root-cause analysis.**